# Where and when did subway service actually change?

Suppose an alert says the N is being rerouted. I know which line to look at, but I still have questions. Which stations lost the N? Did another line keep serving those stations? And did the loss start before the alert came out? I need those answers before I can compare nearby completed pickups.

I use the alerts to find incidents and the reconstructed train records to check where service changed. Then I put those changes on the same taxi-zone and hourly grid as the pickup data. If the archive goes quiet and I can't tell what happened, I carry that uncertainty into the table.

| Question | Source | Why this source |
|---|---|---|
| When was a disruption announced, and why? | MTA Service Alerts (NY Open Data `7kct-peq7`) | The alert archive is the record of what riders were told and the stated cause |
| Which stations lost recorded train service, and when? | Subway Data NYC (daily realtime train archives) | Train-by-train records, reconstructed from the realtime feed, show where trains were recorded |
| Where are the stations, and which zone are they in? | MTA Subway Stations (NY Open Data) + TLC taxi zones | Official station complexes and coordinates, mapped to the zones of the trip records |
| How busy is each station normally? | MTA Subway Hourly Ridership (NY Open Data) | Weights exposure by normal station use (an intensity proxy, not a count of affected people) |

| Term | Meaning in this notebook |
|---|---|
| Alert state | One published alert update, valid until the next update of the same incident |
| Episode | Consecutive alert states of one incident that carry a high-impact effect (reroute, suspension, skipped stops, severe delay) |
| Station complex | A physical station a rider walks into; several platforms (lines) can share one complex |
| Bin | A 15-minute interval |
| Baseline | Train counts at the same local clock time on earlier comparable days without disruption |
| Route loss / access loss | The disrupted route stopped serving a station / no train of any route served it |
| Diversion | Trains of a route appearing where that route does not normally run |
| Placebo window | The same routes and clock time on a quiet earlier week, used to measure false alarms |

The selected results below are from my earlier run. Rebuilding this notebook requires source files beyond the prepared snapshot.


## 1. What do I need before I can measure a disruption?

I keep the choices that define a run in one place. That lets me work through a pilot example without changing the rest of the notebook, then switch to the full study. I'll save the selected settings with the outputs so I can trace a result back to its rules.


In [ ]:
import hashlib
import json
import pickle
import platform
import re
import tarfile
import warnings
import zipfile
from datetime import datetime, timezone
from functools import lru_cache
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import requests
from pandas.tseries.holiday import USFederalHolidayCalendar
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

warnings.filterwarnings("ignore", category=DeprecationWarning, message=".*'generic' unit for NumPy timedelta.*")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 200)


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Open this notebook inside the project containing pyproject.toml.")


PROJECT_ROOT = find_project_root()
print("Project root:", PROJECT_ROOT)

### The configuration

The study covers July 2024–June 2026, with earlier history for comparisons. Fifteen-minute bins give me more detail than hourly counts without reacting to every short interval between trains.

I also separate causes. Equipment failures are more suitable for the main comparison than planned work or weather, which may change how people travel directly. This screening helps narrow the question; it does not make every retained incident unrelated to all other local conditions.


In [ ]:
# Run mode: "pilot" measures only the worked event and its baseline days; "full" measures every study episode
RUN_MODE = "full"

# Retaining the original history and extending through September 2026
LOCAL_TIMEZONE = "America/New_York"
STUDY_START = pd.Timestamp("2024-07-01", tz=LOCAL_TIMEZONE)
STUDY_END = pd.Timestamp("2026-07-01", tz=LOCAL_TIMEZONE)

# Baselines: the same weekday (earlier Sundays for holidays) in up to BASELINE_WEEKS earlier weeks
BASELINE_WEEKS = 6
BASELINE_SEARCH_WEEKS = 12
MIN_BASELINE_DAYS = 3

def shift_local_days(day, days):
    """Moving a New York local midnight by whole calendar days, never by multiples of 24 elapsed hours.
    (Around daylight-saving changes a local day lasts 23 or 25 hours, so adding 24 hours misses midnight.)"""
    naive = pd.Timestamp(day).tz_convert(LOCAL_TIMEZONE).tz_localize(None).normalize()
    return (naive + pd.Timedelta(days=days)).tz_localize(LOCAL_TIMEZONE)


TRAIN_START = shift_local_days(STUDY_START, -7 * (BASELINE_SEARCH_WEEKS + 1))
TRAIN_END = shift_local_days(STUDY_END, 1)
ALERT_START_UTC = shift_local_days(TRAIN_START, -1).tz_convert("UTC")
ALERT_END_UTC = shift_local_days(STUDY_END, 1).tz_convert("UTC")
STUDY_START_UTC = STUDY_START.tz_convert("UTC")
STUDY_END_UTC = STUDY_END.tz_convert("UTC")

# Worked example (June 30, 2024, N rerouted via D and Q)
PILOT_EVENT_ID = 158714
PILOT_DAY = pd.Timestamp("2024-06-30", tz=LOCAL_TIMEZONE)

# Measurement window
BIN = pd.Timedelta(15, unit="min")
PRE_WINDOW = pd.Timedelta(150, unit="min")              # before the incident's first alert
REFERENCE_PERIOD = pd.Timedelta(45, unit="min")         # start of the window: what is abnormal here is background
POST_WINDOW = pd.Timedelta(60, unit="min")              # after the announced end
MAX_MEASURE_WINDOW = pd.Timedelta(12, unit="h")
INCIDENT_START_MAX_LEAD = pd.Timedelta(90, unit="min")  # incident start is capped at this long before onset
MAX_STATE_AGE = pd.Timedelta(3, unit="h")               # assumed end of a final alert state with no later update
BASELINE_EXCLUSION_BUFFER = pd.Timedelta(45, unit="min") # a baseline day is rejected if an episode on the routes is this close
PRE_ALERT_CONNECT = pd.Timedelta(30, unit="min")        # a pre-alert loss counts only if it runs to within this of onset

# Loss rules (calibrated in Section 6)
# Complete loss: a run of empty bins is lost when at least this many trains were expected over it.
# Two pre-specified variants; run the notebook once per variant (outputs go to separate folders).
LOSS_THRESHOLD_VARIANTS = {
    "single_3": {"weekday_daytime": 3.0, "night_or_weekend": 3.0},      # primary
    "period_3_5": {"weekday_daytime": 5.0, "night_or_weekend": 3.0},    # secondary
}
LOSS_THRESHOLD_VARIANT = "period_3_5"   # primary; "single_3" is the robustness run
COMPLETE_LOSS_MIN_EXPECTED = LOSS_THRESHOLD_VARIANTS[LOSS_THRESHOLD_VARIANT]
LOSS_MIN_RUN_BINS = 2              # ... and it lasts at least this many bins
PARTIAL_LOSS_SHARE = 0.5           # partial loss: observed at most this share of expected over a run of low bins
PARTIAL_LOSS_MIN_EXPECTED = 6.0    # ... with at least this many trains expected over the run
BACKGROUND_MIN_EXPECTED = 2.0      # fixed rule for background, independent of the rule being calibrated
NORMALLY_SERVED_SHARE = 0.5        # a route normally serves a complex if it did so on this share of baseline days
# A diversion needs at least this many trains at the unusual station in the window
# (one at night/weekend, when a rerouted line may pass a station only once)
DIVERSION_MIN_TRAINS = {"weekday_daytime": 2, "night_or_weekend": 1}
ACCESS_IGNORES_DIRECTION = False   # False: access loss is judged per platform direction (N/S) at the complex

# Feed health
FEED_OK_RATIO = 0.5
FEED_MIN_EXPECTED = 5
FEED_GROUPS = {
    "1234567S": ["1", "2", "3", "4", "5", "6", "6X", "7", "7X", "GS"],
    "ACE": ["A", "C", "E", "H", "FS"],
    "BDFM": ["B", "D", "F", "FX", "M"],
    "G": ["G"], "JZ": ["J", "Z"], "NQRW": ["N", "Q", "R", "W"], "L": ["L"], "SI": ["SI", "SS"],
}
ROUTE_FEED_GROUP = {route: group for group, routes in FEED_GROUPS.items() for route in routes}

# Alert coverage
PARTIAL_DAY_GAP_HOURS = 8          # longest daytime stretch without updates that flags a partial gap
ALERT_MONTH_SETTLE_DAYS = 3        # a month downloaded sooner than this after it ended may be incomplete

# Cause groups (a research decision: report results under alternatives)
CAUSE_ELIGIBILITY = {
    "signal_problem": "main", "switch_problem": "main", "power_problem": "main", "track_condition": "main",
    "train_mechanical_problem": "main", "door_problem": "main", "debris_or_obstruction": "main",
    "work_equipment": "main", "staffing_or_crew": "main", "cleaning": "main",
    "medical_emergency": "main", "urgent_track_work": "main",
    "police_activity": "robustness", "person_on_tracks": "robustness", "tree_or_vegetation": "robustness",
    "fire_or_smoke": "robustness", "water_condition": "robustness", "unknown": "robustness",
    "weather": "excluded", "planned_maintenance": "excluded", "track_work": "excluded", "special_event": "excluded",
}

# Realtime stop ids missing from the MTA station list, mapped by hand after checking (none needed so far)
MANUAL_STOP_PARENT: dict[str, str] = {}

# Sources
DATASET_ID = "7kct-peq7"
API_URL = f"https://data.ny.gov/resource/{DATASET_ID}.json"
TRAIN_ARCHIVE_URL = "https://subwaydata.nyc/data/subwaydatanyc_{day}_csv.tar.xz"
STATIONS_URL = "https://data.ny.gov/api/views/39hk-dx4f/rows.csv?accessType=DOWNLOAD"
STATIONS_SHA256 = "aa92a66228ae2d17484646bb6eb8444e4edcb45917dd804ed516396f5339646d"
RIDERSHIP_DATASETS = [("2020-01-01", "wujg-7c2s"), ("2025-01-01", "5wq4-mkjj")]  # MTA Subway Hourly Ridership: 2020-2024, 2025+
TAXI_ZONE_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip"
TAXI_ZONE_SHA256 = "f6d711917bb4340f8f644d5366c51665489eb2d426dd1a4a55677721ae5adf17"

# Switches
ALLOW_DOWNLOADS = True
FORCE_REFRESH_ALERT_PERIODS: set[str] = set()
CHECK_ALERT_GAPS_AT_SOURCE = False
KEEP_RAW_TRAIN_ARCHIVES = True
SAVE_OUTPUTS = True
CHECKPOINT_EVERY = 500
N_WORKERS = 10                     # threads for reducing and counting train data (about CPU cores minus one or two)
RUN_TIMING_CHECK = False           # True: time a sample of episodes before the full measurement
PIPELINE_VERSION = "measured-v2"

# Directories
BRONZE_ALERTS_DIR = PROJECT_ROOT / "data" / "bronze" / "mta_service_alerts"
BRONZE_TRAIN_DIR = PROJECT_ROOT / "data" / "bronze" / "subwaydata_nyc"
STATIONS_DIR = PROJECT_ROOT / "data" / "external" / "mta_subway_stations"
TAXI_ZONE_DIR = PROJECT_ROOT / "data" / "external" / "taxi_zones"
RIDERSHIP_DIR = PROJECT_ROOT / "data" / "external" / "mta_hourly_ridership"
TRAIN_REDUCED_DIR = PROJECT_ROOT / "data" / "interim" / "subwaydata_nyc" / "trip_stops_v2"
TRAIN_COUNTS_ROOT = PROJECT_ROOT / "data" / "interim" / "subwaydata_nyc" / "daily_counts_v2"
SILVER_DIR = PROJECT_ROOT / "data" / "silver" / "mta_disruptions"
# Isolating extended outputs and checkpoints from the original saved runs
# SILVER_DIR = (
#     PROJECT_ROOT
#     / "data"
#     / "silver"
#     / "mta_disruptions"
#     / "study_20240701_20260930"
# )

for directory in [BRONZE_ALERTS_DIR, BRONZE_TRAIN_DIR, STATIONS_DIR, TAXI_ZONE_DIR, RIDERSHIP_DIR,
                  TRAIN_REDUCED_DIR, TRAIN_COUNTS_ROOT, SILVER_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

retry_policy = Retry(total=5, connect=5, read=5, backoff_factor=1,
                     status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"])
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retry_policy))

### Helper functions

The helper functions handle downloads, clocks, and overlapping intervals. The time conversions matter here: an alert, a train stop, and a pickup need to refer to the same real hour before their patterns can be compared.


In [ ]:
def sha256_of(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


class NotAtSource(Exception):
    """The source has no file for this request (HTTP 404)."""


def fetch_cached_file(url, destination, validator=None, expected_sha256=None):
    """Downloading a missing file to a temporary path and validating it before caching."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        if not ALLOW_DOWNLOADS:
            raise FileNotFoundError(f"Missing cache: {destination}")
        temporary = destination.with_suffix(destination.suffix + ".part")
        try:
            response = session.get(url, timeout=180)
            if response.status_code == 404:
                raise NotAtSource(url)
            response.raise_for_status()
            assert response.content, "The source returned an empty file."
            temporary.write_bytes(response.content)
            if validator:
                validator(temporary)
            if expected_sha256:
                assert sha256_of(temporary) == expected_sha256, "Source checksum mismatch."
            temporary.replace(destination)
        finally:
            temporary.unlink(missing_ok=True)
    if validator:
        validator(destination)
    if expected_sha256:
        assert sha256_of(destination) == expected_sha256, "Cached file checksum mismatch."
    return destination


def local_dates(start_local, end_local):
    """Local calendar dates in [start, end)."""
    return pd.date_range(start_local.normalize(), end_local.normalize(), freq="D", inclusive="left")


def to_local_naive(timestamps):
    """New York wall-clock time without a time zone (the TLC trip-record convention)."""
    return pd.to_datetime(timestamps).dt.tz_convert(LOCAL_TIMEZONE).dt.tz_localize(None)


def localize_wall_clock(naive):
    """A New York wall-clock time as a UTC timestamp (first occurrence when ambiguous, shifted forward when skipped)."""
    return pd.Timestamp(naive).tz_localize(LOCAL_TIMEZONE, ambiguous=True, nonexistent="shift_forward").tz_convert("UTC")


def intervals_to_hourly(frame, keys, start_col="start", end_col="end"):
    """Unioning intervals per key and counting covered minutes in each UTC hour (vectorized)."""
    hour = pd.Timedelta(1, unit="h")
    output_columns = [*keys, "hour_utc", "minutes"]

    data = (frame.loc[frame[end_col] > frame[start_col], [*keys, start_col, end_col]]
            .rename(columns={start_col: "start", end_col: "end"})
            .sort_values([*keys, "start"], kind="stable")
            .reset_index(drop=True))
    if data.empty:
        empty = pd.DataFrame(columns=output_columns)
        empty["hour_utc"] = pd.to_datetime(empty["hour_utc"], utc=True)
        return empty

    # Merging overlapping or touching intervals within each key
    end_ns = data["end"].dt.tz_convert("UTC").dt.tz_localize(None).astype("int64")
    start_ns = data["start"].dt.tz_convert("UTC").dt.tz_localize(None).astype("int64")
    groups = [data[k] for k in keys]
    latest_earlier_end = end_ns.groupby(groups, sort=False, dropna=False).cummax()
    latest_earlier_end = latest_earlier_end.groupby(groups, sort=False, dropna=False).shift(1)
    data["block"] = (latest_earlier_end.isna() | (start_ns > latest_earlier_end)).cumsum()
    merged = data.groupby("block", sort=False).agg(
        **{k: (k, "first") for k in keys}, start=("start", "min"), end=("end", "max"))

    # Splitting each merged interval into the UTC hours it touches
    first_hour = merged["start"].dt.floor("h")
    last_hour = (merged["end"] - pd.Timedelta(1, unit="ns")).dt.floor("h")
    hours_touched = ((last_hour - first_hour) // hour + 1).astype(int)
    pieces = merged.loc[merged.index.repeat(hours_touched)].copy()
    pieces["hour_utc"] = first_hour.loc[pieces.index] + pieces.groupby(level=0).cumcount().to_numpy() * hour
    piece_end = pieces["end"].where(pieces["end"] < pieces["hour_utc"] + hour, pieces["hour_utc"] + hour)
    piece_start = pieces["start"].where(pieces["start"] > pieces["hour_utc"], pieces["hour_utc"])
    pieces["minutes"] = (piece_end - piece_start).dt.total_seconds() / 60

    output = (pieces.groupby([*keys, "hour_utc"], sort=False, dropna=False)["minutes"].sum()
              .reset_index()[output_columns])
    assert output["minutes"].gt(0).all() and output["minutes"].le(60 + 1e-6).all()
    return output

## 2. What did riders hear about the disruption?

I'll start with the alert archive itself. I need to know whether it covers the study period, which clock it uses, and how repeated updates become episodes. Otherwise later measurements could be precise calculations attached to the wrong hours.


### Loading the archive

I save the archive month by month and reuse those files. That makes the collection easier to resume. A cached month downloaded before it ended needs extra care, though, because rerunning against that file would keep the original incomplete version.


In [ ]:
alert_month_log = []


def download_alert_month(year, month, force_refresh=False, page_size=50_000):
    """Downloading (or loading the cached copy of) one UTC month of NYCT Subway alerts."""
    month_start = pd.Timestamp(year=year, month=month, day=1)
    month_end = month_start + pd.offsets.MonthBegin(1)
    output_path = BRONZE_ALERTS_DIR / f"mta_subway_service_alerts_{year}_{month:02d}.json"

    if output_path.exists() and not force_refresh:
        payload = json.loads(output_path.read_text(encoding="utf-8"))
    else:
        if not ALLOW_DOWNLOADS:
            raise FileNotFoundError(f"Missing monthly cache: {output_path}")
        records, offset = [], 0
        while True:
            response = session.get(API_URL, timeout=120, params={
                "$where": (f"agency = 'NYCT Subway' AND date >= '{month_start:%Y-%m-%dT%H:%M:%S}.000' "
                           f"AND date < '{month_end:%Y-%m-%dT%H:%M:%S}.000'"),
                "$order": "date, event_id, update_number, alert_id",
                "$limit": page_size,
                "$offset": offset,
            })
            response.raise_for_status()
            page = response.json()
            records.extend(page)
            if len(page) < page_size:
                break
            offset += page_size
        payload = {"source_url": API_URL, "dataset_id": DATASET_ID,
                   "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
                   "record_count": len(records), "records": records}
        output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

    # Integrity: the stored count must match the records, and the month should have been complete when fetched
    if "record_count" in payload:
        assert payload["record_count"] == len(payload["records"]), f"{output_path.name} is truncated"
    fetched = pd.to_datetime(payload.get("fetched_at_utc"), utc=True) if payload.get("fetched_at_utc") else pd.NaT
    alert_month_log.append({
        "period": f"{year}_{month:02d}", "records": len(payload["records"]), "fetched_at_utc": fetched,
        "possibly_incomplete": bool(pd.notna(fetched) and fetched < (month_end + pd.Timedelta(days=ALERT_MONTH_SETTLE_DAYS)).tz_localize("UTC")),
        "file_sha256": sha256_of(output_path),
    })
    return pd.DataFrame(payload["records"])


def clean_alert_month(raw, year, month):
    """Typing one month and reading its clock as UTC."""
    columns = ["alert_id", "event_id", "update_number", "date", "agency",
               "status_label", "affected", "header", "description"]
    if raw.empty:
        raise ValueError(f"Empty alert month {year}-{month:02d}")
    if "description" not in raw.columns:
        raw = raw.assign(description=pd.NA)
    missing = set(columns) - set(raw.columns)
    if missing:
        raise ValueError(f"Missing columns {sorted(missing)} in {year}-{month:02d}")

    alerts = raw[columns].copy()
    for column in ["alert_id", "event_id", "update_number"]:
        alerts[column] = pd.to_numeric(alerts[column], errors="raise").astype("int64")
    alerts["date_raw"] = alerts["date"].astype("string")
    alerts["date"] = pd.to_datetime(alerts["date_raw"], utc=True, errors="raise")
    alerts["date_ny"] = alerts["date"].dt.tz_convert(LOCAL_TIMEZONE)
    alerts["source_period"] = f"{year}_{month:02d}"

    month_start = pd.Timestamp(year=year, month=month, day=1, tz="UTC")
    assert set(alerts["agency"].dropna()) == {"NYCT Subway"}
    assert alerts["date"].ge(month_start).all() and alerts["date"].lt(month_start + pd.offsets.MonthBegin(1)).all()
    assert alerts["alert_id"].is_unique
    return alerts


alert_periods = pd.period_range(ALERT_START_UTC.tz_localize(None), ALERT_END_UTC.tz_localize(None), freq="M")
alerts = pd.concat(
    [clean_alert_month(download_alert_month(p.year, p.month, f"{p.year}_{p.month:02d}" in FORCE_REFRESH_ALERT_PERIODS),
                       p.year, p.month) for p in alert_periods],
    ignore_index=True,
)
alerts = alerts.loc[alerts["date"].ge(ALERT_START_UTC) & alerts["date"].lt(ALERT_END_UTC)]
alerts = alerts.sort_values(["event_id", "date", "update_number", "alert_id"]).reset_index(drop=True)
assert alerts["alert_id"].is_unique

alert_month_log = pd.DataFrame(alert_month_log)
possibly_incomplete = alert_month_log.loc[alert_month_log["possibly_incomplete"], "period"].tolist()
if possibly_incomplete:
    warnings.warn(f"Months fetched before they were complete (add to FORCE_REFRESH_ALERT_PERIODS): {possibly_incomplete}")

display(pd.Series({
    "alert_rows": len(alerts), "events": alerts["event_id"].nunique(),
    "first_update_ny": alerts["date_ny"].min(), "last_update_ny": alerts["date_ny"].max(),
    "months_possibly_incomplete": len(possibly_incomplete),
}, name="value").to_frame())

### Was service quiet, or was the archive missing?

The coverage check reveals a pattern that would be easy to miss: some apparently quiet periods are whole missing UTC days. Those are not evidence of normal subway service. I'll mark the gaps as unknown so they cannot quietly become clean comparison hours later.


In [ ]:
# Daily alert coverage on New York calendar days: empty days and days with a long daytime silence
study_days = local_dates(STUDY_START, STUDY_END)
alert_coverage_daily = (
    alerts["date_ny"].dt.normalize().value_counts()
    .reindex(study_days, fill_value=0)
    .rename_axis("day_ny").reset_index(name="alert_updates")
)
alert_coverage_daily["alerts_covered"] = alert_coverage_daily["alert_updates"].gt(0)

# Longest stretch between 06:00 and 24:00 without any update
local_hours = (alerts.assign(day_ny=alerts["date_ny"].dt.normalize(),
                             hour=alerts["date_ny"].dt.hour + alerts["date_ny"].dt.minute / 60)
               .loc[lambda f: f["hour"].ge(6)])
longest_gap = {}
for day, hours in local_hours.groupby("day_ny")["hour"]:
    points = np.sort(np.concatenate([[6.0], hours.to_numpy(), [24.0]]))
    longest_gap[day] = float(np.diff(points).max())
alert_coverage_daily["longest_daytime_gap_hours"] = alert_coverage_daily["day_ny"].map(longest_gap).fillna(18.0)
alert_coverage_daily["partial_gap"] = (alert_coverage_daily["alerts_covered"]
                                       & alert_coverage_daily["longest_daytime_gap_hours"].ge(PARTIAL_DAY_GAP_HOURS))

alert_gap_days = set(alert_coverage_daily.loc[~alert_coverage_daily["alerts_covered"], "day_ny"].dt.date)
alert_partial_gap_days = set(alert_coverage_daily.loc[alert_coverage_daily["partial_gap"], "day_ny"].dt.date)

# The archive is partitioned by UTC day: a UTC day with no rows is a source gap covering that exact 24 hours
utc_days = pd.date_range(STUDY_START_UTC.floor("D") - pd.Timedelta(days=1), STUDY_END_UTC.floor("D"), freq="D")
utc_day_counts = alerts["date"].dt.floor("D").value_counts().reindex(utc_days, fill_value=0)
source_gap_utc_days = utc_day_counts.index[utc_day_counts.eq(0)]
alert_gap_intervals = pd.DataFrame({"start": source_gap_utc_days, "end": source_gap_utc_days + pd.Timedelta(days=1)})


def overlaps_alert_gap(start, end):
    """True when [start, end) UTC overlaps a UTC day missing from the archive."""
    return bool(((alert_gap_intervals["start"] < end) & (alert_gap_intervals["end"] > start)).any())


print("UTC days missing from the archive:", len(source_gap_utc_days))
display(pd.Series(source_gap_utc_days.strftime("%Y-%m-%d"), name="missing_utc_day").to_frame().T)

print("Study days without alert updates:", len(alert_gap_days))
print(f"Study days with a daytime silence of {PARTIAL_DAY_GAP_HOURS}+ hours:", len(alert_partial_gap_days))
display(alert_coverage_daily.loc[~alert_coverage_daily["alerts_covered"] | alert_coverage_daily["partial_gap"]])

# Optional: confirming at the source that gap days are empty there too
if CHECK_ALERT_GAPS_AT_SOURCE and alert_gap_days:
    rows = []
    for day in sorted(alert_gap_days):
        start = pd.Timestamp(day, tz=LOCAL_TIMEZONE).tz_convert("UTC")
        end = shift_local_days(pd.Timestamp(day, tz=LOCAL_TIMEZONE), 1).tz_convert("UTC")
        response = session.get(API_URL, timeout=60, params={
            "$select": "count(*) AS n",
            "$where": f"agency = 'NYCT Subway' AND date >= '{start:%Y-%m-%dT%H:%M:%S}' AND date < '{end:%Y-%m-%dT%H:%M:%S}'",
        })
        response.raise_for_status()
        rows.append({"day_ny": day, "source_updates": int(response.json()[0]["n"])})
    display(pd.DataFrame(rows))

UTC days missing from the archive: 35


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34
missing_utc_day,2024-07-12,2024-07-13,2024-08-31,2024-09-30,2024-10-31,2024-11-18,2024-11-19,2024-11-20,2024-11-21,2024-11-22,2024-11-23,2024-11-24,2024-11-25,2024-11-26,2024-11-27,2024-11-28,2024-11-29,2024-11-30,2025-02-28,2025-03-31,2025-04-30,2025-05-31,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30


Study days without alert updates: 13
Study days with a daytime silence of 8+ hours: 22


,day_ny,alert_updates,alerts_covered,longest_daytime_gap_hours,partial_gap
11,2024-07-12 00:00:00-04:00,0,False,18.000000,False
12,2024-07-13 00:00:00-04:00,19,True,14.200000,True
61,2024-08-31 00:00:00-04:00,6,True,14.616667,True
91,2024-09-30 00:00:00-04:00,26,True,14.033333,True
122,2024-10-31 00:00:00-04:00,13,True,14.050000,True
140,2024-11-18 00:00:00-05:00,0,False,18.000000,False
141,2024-11-19 00:00:00-05:00,0,False,18.000000,False
142,2024-11-20 00:00:00-05:00,0,False,18.000000,False
143,2024-11-21 00:00:00-05:00,0,False,18.000000,False
144,2024-11-22 00:00:00-05:00,0,False,18.000000,False


### Which clock does the archive use?

The timestamps arrive without a time-zone label. A matched public announcement points to UTC, and the summer-winter activity profiles support that interpretation. I'll use UTC for reading the archive and convert to New York time when comparing local travel patterns.


In [ ]:
# Clock check across daylight saving time: with a true UTC source, the local profiles line up without a shift
alerts_by_local_hour = (
    alerts.loc[alerts["date"].ge(STUDY_START_UTC) & alerts["date"].lt(STUDY_END_UTC)]
    .assign(month=lambda f: f["date_ny"].dt.strftime("%Y-%m"), hour=lambda f: f["date_ny"].dt.hour)
    .groupby(["month", "hour"]).size().unstack(fill_value=0)
    .reindex(columns=range(24), fill_value=0)
)
is_dst_month = pd.Series([pd.Timestamp(f"{m}-15", tz=LOCAL_TIMEZONE).dst() != pd.Timedelta(0)
                          for m in alerts_by_local_hour.index], index=alerts_by_local_hour.index)
hour_profile = (alerts_by_local_hour.div(alerts_by_local_hour.sum(axis=1), axis=0)
                .groupby(is_dst_month).mean().T.rename(columns={True: "dst_share", False: "standard_share"}))
if {"dst_share", "standard_share"} <= set(hour_profile.columns):
    print("Correlation as recorded:", round(hour_profile["dst_share"].corr(hour_profile["standard_share"]), 3),
          "| DST shifted one hour later:", round(hour_profile["dst_share"].shift(-1).corr(hour_profile["standard_share"]), 3),
          "| one hour earlier:", round(hour_profile["dst_share"].shift(1).corr(hour_profile["standard_share"]), 3))
else:
    print("The study window does not contain both DST and standard-time months.")

Correlation as recorded: 0.984 | DST shifted one hour later: 0.871 | one hour earlier: 0.786


### Which alert updates should count?

Plain delays can describe many situations, so I don't treat every delay update as a high-impact disruption. I'll use the structured effect labels and retain the last correction within a minute. That gives each update time one state before building episodes.


In [ ]:
STATUS_TO_EFFECT = {
    "severe-delays": "severe_delay", "part-suspended": "partial_suspension", "suspended": "full_suspension",
    "cancellations": "cancellation", "stops-skipped": "stop_skipping", "reroute": "reroute",
    "express-to-local": "express_to_local", "boarding-change": "boarding_change",
}
KNOWN_STATUS_COMPONENTS = set(STATUS_TO_EFFECT) | {"delays", "information-outage", "special-notice", "station-notice"}
# Ordered by severity: used to pick one primary effect
EFFECT_ORDER = ["full_suspension", "partial_suspension", "reroute", "stop_skipping", "severe_delay",
                "express_to_local", "boarding_change", "cancellation"]
EPISODE_EFFECTS = {"severe_delay", "partial_suspension", "full_suspension", "stop_skipping", "reroute"}

ALTERNATIVE_SERVICE_SENTENCE = re.compile(
    r"^\s*(?:for service|consider|take |use |transfer|travel alternatives?|see a station agent|view trains)", re.I)


def incident_text(header, description):
    """Alert text without standing reminders or travel advice."""
    text = "\n".join(str(v) for v in (header, description) if pd.notna(v))
    text = re.split(r"\breminders?\s*:", text, flags=re.I)[0]
    pieces = [p.strip() for p in re.split(r"\n+|(?<=[.!?])\s+", text)]
    return " ".join(p for p in pieces if p and not ALTERNATIVE_SERVICE_SENTENCE.match(p))


components = alerts["status_label"].str.split("|").explode().str.strip()
unknown_components = sorted(set(components.dropna()) - KNOWN_STATUS_COMPONENTS)
assert not unknown_components, f"Classify these status components first: {unknown_components}"


def detect_effects(status_label, header, description):
    """Structured status effects, plus severe delays stated in the incident's own text."""
    found = {STATUS_TO_EFFECT[c.strip()] for c in str(status_label).split("|") if c.strip() in STATUS_TO_EFFECT}
    if re.search(r"\bsevere delays?\b", incident_text(header, description), re.I):
        found.add("severe_delay")
    return tuple(e for e in EFFECT_ORDER if e in found)


# Keeping the final revision per event-minute and recording when each state was replaced
alerts_sorted = alerts.sort_values(["event_id", "date", "update_number", "alert_id"])
timeline = alerts_sorted.drop_duplicates(["event_id", "date"], keep="last").reset_index(drop=True)
same_minute_revisions = len(alerts_sorted) - len(timeline)

timeline["next_update_time"] = timeline.groupby("event_id")["date"].shift(-1)
timeline["incident_text"] = [incident_text(h, d) for h, d in zip(timeline["header"], timeline["description"])]
timeline["explicit_effects"] = [detect_effects(*row) for row in
                                timeline[["status_label", "header", "description"]].itertuples(index=False)]
timeline["episode_effects"] = timeline["explicit_effects"].map(lambda es: tuple(e for e in es if e in EPISODE_EFFECTS))

has_next = timeline["next_update_time"].notna()
assert (timeline.loc[has_next, "next_update_time"] > timeline.loc[has_next, "date"]).all()

print("Same-minute revisions removed:", same_minute_revisions, "| States kept:", len(timeline))
display(timeline["explicit_effects"].explode().dropna().value_counts().to_frame("states"))

Same-minute revisions removed: 1082 | States kept: 103186


,states
explicit_effects,
stop_skipping,7487
reroute,4286
partial_suspension,3643
express_to_local,3453
severe_delay,1422
full_suspension,357
boarding_change,257
cancellation,151


### When does one episode end and another begin?

An episode opens when a high-impact effect appears and closes when a later update removes it. If it returns, that becomes another episode. Some updates simply stop, leaving no observed closing state. I'll keep those episodes marked as censored rather than pretend their recovery time is known.


In [ ]:
def build_episodes(states):
    """An episode runs while any episode effect is present in consecutive states of one event."""
    rows = []
    for event_id, group in states.groupby("event_id", sort=False):
        current = None
        for s in group.itertuples(index=False):
            if s.episode_effects and current is None:
                current = {"event_id": int(event_id), "onset_alert_id": int(s.alert_id), "onset_time": s.date,
                           "onset_effects": s.episode_effects, "observed": set(s.episode_effects),
                           "active_alert_ids": [int(s.alert_id)], "last_active_time": s.date}
            elif s.episode_effects:
                current["observed"].update(s.episode_effects)
                current["active_alert_ids"].append(int(s.alert_id))
                current["last_active_time"] = s.date
            elif current is not None:
                rows.append({**current, "end_alert_id": int(s.alert_id), "end_time": s.date})
                current = None
        if current is not None:
            rows.append({**current, "end_alert_id": pd.NA, "end_time": pd.NaT})

    episodes = pd.DataFrame(rows).sort_values(["event_id", "onset_time"]).reset_index(drop=True)
    episodes["observed_effects"] = episodes.pop("observed").map(lambda es: tuple(e for e in EFFECT_ORDER if e in es))
    episodes["active_alert_ids"] = episodes["active_alert_ids"].map(tuple)
    episodes["end_alert_id"] = episodes["end_alert_id"].astype("Int64")
    episodes["end_time"] = pd.to_datetime(episodes["end_time"], utc=True)
    episodes["is_right_censored"] = episodes["end_time"].isna()
    episodes["episode_number"] = episodes.groupby("event_id").cumcount() + 1
    episodes["episode_key"] = episodes["event_id"].astype(str) + "-" + episodes["episode_number"].astype(str)
    episodes["alert_end"] = episodes["end_time"].fillna(episodes["last_active_time"] + MAX_STATE_AGE)
    assert episodes["episode_key"].is_unique
    return episodes


candidate_event_ids = set(timeline.loc[timeline["episode_effects"].map(bool), "event_id"])
candidate_history = timeline.loc[timeline["event_id"].isin(candidate_event_ids)].copy()
episodes = build_episodes(candidate_history)

# Incident start: the event's first update for its first episode (capped), later episodes at their own onset
event_first_update = candidate_history.groupby("event_id")["date"].min()
episodes["incident_start_uncapped"] = episodes["event_id"].map(event_first_update).where(
    episodes["episode_number"].eq(1), episodes["onset_time"])
episodes["incident_start"] = episodes["incident_start_uncapped"].where(
    episodes["incident_start_uncapped"] >= episodes["onset_time"] - INCIDENT_START_MAX_LEAD,
    episodes["onset_time"] - INCIDENT_START_MAX_LEAD)
episodes["incident_start_capped"] = episodes["incident_start"] > episodes["incident_start_uncapped"]

holidays = set(USFederalHolidayCalendar().holidays(TRAIN_START.tz_localize(None), TRAIN_END.tz_localize(None)).date)


def time_period(timestamp):
    """Analysis period of a moment in New York time."""
    local = timestamp.tz_convert(LOCAL_TIMEZONE)
    if local.dayofweek >= 5 or local.date() in holidays:
        return "weekend_day" if 6 <= local.hour < 22 else "weekend_night"
    if 6 <= local.hour < 10:
        return "am_peak"
    if 10 <= local.hour < 16:
        return "midday"
    if 16 <= local.hour < 20:
        return "pm_peak"
    return "evening" if local.hour >= 20 else "late_night"


def broad_period(period):
    """Two-way split used by the loss threshold: weekday daytime (06:00-19:59) versus everything else."""
    return "weekday_daytime" if period in ("am_peak", "midday", "pm_peak") else "night_or_weekend"


episodes["onset_day_ny"] = episodes["onset_time"].dt.tz_convert(LOCAL_TIMEZONE).dt.date
episodes["period"] = episodes["onset_time"].map(time_period)
episodes["weekend_or_holiday"] = episodes["period"].str.startswith("weekend")
episodes["broad_period"] = episodes["period"].map(broad_period)
episodes["primary_effect"] = episodes["observed_effects"].str[0]
episodes["primary_onset_effect"] = episodes["onset_effects"].str[0]
episodes["within_study_window"] = episodes["onset_time"].ge(STUDY_START_UTC) & episodes["onset_time"].lt(STUDY_END_UTC)
episodes["on_alert_gap_day"] = [
    overlaps_alert_gap(start - pd.Timedelta(hours=6), end + pd.Timedelta(hours=6)) or day in alert_partial_gap_days
    for start, end, day in zip(episodes["incident_start"], episodes["alert_end"], episodes["onset_day_ny"])]


# Why an episode never closed
def censor_reason(ep):
    if not ep.is_right_censored:
        return None
    if ep.last_active_time >= ALERT_END_UTC - pd.Timedelta(days=1):
        return "data_end"
    if overlaps_alert_gap(ep.last_active_time, ep.last_active_time + pd.Timedelta(days=1)):
        return "archive_gap"
    day = ep.last_active_time.tz_convert(LOCAL_TIMEZONE).date()
    if day in alert_partial_gap_days:
        return "archive_gap"
    return "no_end_update"


episodes["censor_reason"] = [censor_reason(ep) for ep in episodes.itertuples(index=False)]

display(episodes.loc[episodes["within_study_window"]]
        .agg({"episode_key": "size", "is_right_censored": "sum", "incident_start_capped": "sum"})
        .rename({"episode_key": "episodes"}).to_frame("study"))
display(episodes.loc[episodes["within_study_window"], "censor_reason"].value_counts().to_frame("censored_episodes"))

,study
episodes,8381
is_right_censored,400
incident_start_capped,62


,censored_episodes
censor_reason,
no_end_update,379
archive_gap,21


### Which causes belong in my comparison?

The cause affects which incidents I can use later, so I read it from the onset and nearby updates before relying on broader episode text. Specific matches take priority, and unmatched cases remain visible. I'll inspect examples instead of assuming the text rules have captured every wording.


In [ ]:
# Ordered by priority: specific causes before broad ones. All matches are recorded.
CAUSE_PATTERNS = [
    ("signal_problem", r"\bsignal (?:problem|malfunction|issue)|\bsignal(?:ing)? system\b"),
    ("switch_problem", r"\bswitch (?:problem|malfunction|issue)|\bunable to move (?:a |the )?switch(?:es)?\b"),
    ("power_problem", r"\b(?:power|electrical) (?:problem|issue|outage)|\bloss (?:of|a) (?:third[- ]rail )?power\b|\bthird[- ]rail power\b|\blighting\b"),
    ("track_condition", r"\b(?:track|rail) (?:condition|problem|defect|repair)|\bbroken rail"),
    ("special_event", r"\bparade\b|\bmarathon\b|\bcelebration\b|\bspecial event\b|\bchampionship\b"
                      r"|\bceremony\b|\bdemonstration\b|\bprotest\b"),
    ("urgent_track_work", r"\b(?:urgent|emergency) (?:track |rail |station )?(?:maintenance|repairs?|work)\b"),
    ("water_condition", r"\bwater main\b|\bwater condition\b|\bwater (?:from|on) the tracks\b"),
    ("fire_or_smoke", r"\bFDNY\b|\bfire\b|\bsmoke\b"),
    ("medical_emergency", r"\bEMS\b|\bmedical\b|\bsick (?:customer|person)"),
    ("person_on_tracks", r"\b(?:person|someone|customer)\b[^.]*\btracks?\b|\bunauthorized person"
                         r"|\bstruck by a train\b|\b(?:person|someone|customer) (?:was |who was )?struck\b"),
    ("police_activity", r"\bNYPD\b|\bpolice\b|\binvestigation\b|\bdisruptive\b|\bdisorderly\b|\bunruly\b"),
    ("train_mechanical_problem", r"\bmechanical\b|\bbrakes?\b|\bbraking\b|\bunable to move (?:a |the )?trains?\b"
                                 r"|\bremov(?:e|ing) (?:a|the) train (?:from service|car)\b"
                                 r"|\bstalled train\b|\btrain (?:that|which) (?:needs|needed) (?:maintenance|repairs?)\b"),
    ("door_problem", r"\bdoor (?:problem|malfunction|issue)"),
    ("debris_or_obstruction", r"\bdebris\b|\bobstruction\b"),
    ("weather", r"\b(?:weather|flooding|snow|ice|storm|heat)\b"),
    ("tree_or_vegetation", r"\btrees?\b|\bbranch(?:es)?\b"),
    ("work_equipment", r"\bwork (?:train|equipment)\b"),
    ("staffing_or_crew", r"\b(?:crew|staffing|operator availability)\b"),
    ("cleaning", r"\bclean(?:ed|ing)?\b"),
    ("planned_maintenance", r"\bplanned work\b|\bscheduled (?:work|maintenance)\b"),
    ("track_work", r"\brail replacements?\b|\breplac(?:e|es|ing) (?:a |the )?rails?\b|\btrack (?:maintenance|work)\b"
                   r"|\bconstruction\b|\bperform(?:ing)? (?:track|rail)\b|\b(?:signal|switch) maintenance\b|\bdrop(?:ping)? rails?\b"),
]
assert {name for name, _ in CAUSE_PATTERNS} | {"unknown"} == set(CAUSE_ELIGIBILITY), "Cause list and groups differ"


def matched_causes(text):
    return tuple(name for name, pattern in CAUSE_PATTERNS if re.search(pattern, text, re.I))


# Cause from the onset state and the state before it; if none, from every state of the episode
states_by_event = {e: g.reset_index(drop=True) for e, g in candidate_history.groupby("event_id")}
cause_rows = []
for ep in episodes.itertuples(index=False):
    states = states_by_event[ep.event_id]
    position = states.index[states["alert_id"].eq(ep.onset_alert_id)][0]
    found = matched_causes(" ".join(states.iloc[max(0, position - 1): position + 1]["incident_text"]))
    source = "onset"
    if not found:
        episode_states = states.loc[states["alert_id"].isin(ep.active_alert_ids)]
        found = matched_causes(" ".join(episode_states["incident_text"]))
        source = "episode" if found else "none"
    cause_rows.append({"episode_key": ep.episode_key, "causes_matched": found,
                       "cause": found[0] if found else "unknown", "cause_source": source})

episodes = episodes.merge(pd.DataFrame(cause_rows), on="episode_key", how="left", validate="one_to_one")
episodes["multiple_causes"] = episodes["causes_matched"].str.len().gt(1)
episodes["cause_eligibility"] = episodes["cause"].map(CAUSE_ELIGIBILITY)
assert episodes["cause_eligibility"].notna().all()

study = episodes.loc[episodes["within_study_window"]]
display(study.groupby(["cause_eligibility", "cause"]).size().rename("study_episodes").to_frame())
display(study["cause_source"].value_counts().to_frame("study_episodes"))
print("Study episodes matching more than one cause:", int(study["multiple_causes"].sum()))

These are the cases the cause rules still cannot classify. Keeping a sample here lets me see what the unknown group contains and whether the patterns need another revision.


In [ ]:
# Reviewing a reproducible sample of episodes without a cause, to extend CAUSE_PATTERNS
unknown_review = (study.loc[study["cause"].eq("unknown"), ["episode_key", "onset_alert_id"]]
                  .merge(candidate_history[["alert_id", "incident_text"]], left_on="onset_alert_id", right_on="alert_id"))
display(unknown_review.sample(n=min(25, len(unknown_review)), random_state=42)[["episode_key", "incident_text"]])

### Which routes were disrupted, and which were alternatives?

The routes mentioned in an alert do not all play the same role. In the N example, some are alternatives receiving rerouted trains rather than lines losing service. I'll separate those roles before using route names to define exposure or exclude historical dates.


In [ ]:
# Route roles: routes named after "via" are alternatives; the rest of the affected field are disrupted routes
VIA_ROUTES = re.compile(r"\bvia\s+(?:the\s+)?((?:[A-Z0-9]{1,3})(?:(?:\s*,\s*|\s+or\s+|\s+and\s+|\s*/\s*)[A-Z0-9]{1,3})*)")
text_by_alert = candidate_history.set_index("alert_id")["incident_text"]
affected_by_alert = candidate_history.set_index("alert_id")["affected"]


def affected_symbols(alert_ids):
    return {token.strip() for alert_id in alert_ids
            for token in str(affected_by_alert.get(alert_id, "")).split("|")
            if token.strip() and token.strip().lower() != "nan"}


def via_symbols(alert_ids):
    return {token for alert_id in alert_ids
            for match in VIA_ROUTES.finditer(str(text_by_alert.get(alert_id, "")))
            for token in re.findall(r"[A-Z0-9]{1,3}", match.group(1))}


episodes["affected_symbols"] = episodes["active_alert_ids"].map(lambda ids: tuple(sorted(affected_symbols(ids))))
episodes["alternative_symbols"] = [tuple(sorted(via_symbols(ids) & set(symbols)))
                                   for ids, symbols in zip(episodes["active_alert_ids"], episodes["affected_symbols"])]
episodes["disrupted_symbols"] = [tuple(sorted(set(a) - set(b))) or a
                                 for a, b in zip(episodes["affected_symbols"], episodes["alternative_symbols"])]
episodes["episode_text"] = episodes["active_alert_ids"].map(lambda ids: " ".join(str(text_by_alert.get(i, "")) for i in ids))

display(episodes.loc[episodes["event_id"].eq(PILOT_EVENT_ID),
                     ["episode_key", "affected_symbols", "disrupted_symbols", "alternative_symbols", "cause"]])

## 3. Which taxi zones could a station disruption affect?

To connect service loss to pickups, I need a geographic bridge: recorded stop to station complex, then station complex to taxi zone. Normal ridership adds another layer for describing the possible intensity of the loss.


### Official station complexes

The official station list gives me IDs, complexes, coordinates, and usual routes. I'll keep a checksum with the run so a changed reference cannot silently alter the results. Historical suitability still needs checking when a station or route changes over time.


In [ ]:
# MTA Subway Stations (NY Open Data): official complexes, coordinates, borough, daytime routes
stations_path = fetch_cached_file(STATIONS_URL, STATIONS_DIR / "mta_subway_stations.csv", expected_sha256=STATIONS_SHA256)
mta_stations = pd.read_csv(stations_path, dtype=str)

BOROUGH_NAMES = {"M": "Manhattan", "Bk": "Brooklyn", "Bx": "Bronx", "Q": "Queens", "SI": "Staten Island"}
stations = pd.DataFrame({
    "station_id": mta_stations["GTFS Stop ID"].str.strip(),
    "station_complex_id": mta_stations["Complex ID"].str.strip(),
    "station_name": mta_stations["Stop Name"],
    "station_borough": mta_stations["Borough"].map(BOROUGH_NAMES).fillna(mta_stations["Borough"]),
    "daytime_routes": mta_stations["Daytime Routes"].fillna("").str.split().map(tuple),
    "north_label": mta_stations["North Direction Label"],
    "south_label": mta_stations["South Direction Label"],
    "lat": mta_stations["GTFS Latitude"].astype(float),
    "lon": mta_stations["GTFS Longitude"].astype(float),
})
assert stations["station_id"].is_unique, "A GTFS Stop ID appears twice in the station list"
assert stations[["lat", "lon"]].notna().all().all()

station_to_complex = stations.set_index("station_id")["station_complex_id"].to_dict()
complex_names = stations.groupby("station_complex_id")["station_name"].agg(lambda n: " / ".join(sorted(set(n))))

for realtime_parent, listed_station in MANUAL_STOP_PARENT.items():
    assert listed_station in station_to_complex, f"Manual mapping target {listed_station} is not in the station list"

# Daytime routes per complex, used for announced exposure
route_complexes = (stations.explode("daytime_routes").dropna(subset=["daytime_routes"])
                   .groupby("daytime_routes")["station_complex_id"].agg(frozenset).to_dict())

print("Stations:", len(stations), "| Complexes:", stations["station_complex_id"].nunique(),
      "| Routes in daytime lists:", len(route_complexes))

Stations: 496 | Complexes: 445 | Routes in daytime lists: 24


### What if one station complex spans two zones?

I place the station points inside TLC polygons. Some complexes span two zones, which means the same complex can contribute to both. That is reasonable for local exposure, but I cannot then add zone totals and interpret them as distinct stations or passengers.


In [ ]:
taxi_zip = fetch_cached_file(TAXI_ZONE_URL, TAXI_ZONE_DIR / "taxi_zones.zip", expected_sha256=TAXI_ZONE_SHA256)
taxi_dir = TAXI_ZONE_DIR / "extracted"
if not any(taxi_dir.rglob("taxi_zones.shp")):
    with zipfile.ZipFile(taxi_zip) as archive:
        archive.extractall(taxi_dir)

taxi_zones = gpd.read_file(next(taxi_dir.rglob("taxi_zones.shp")))
taxi_zones["LocationID"] = taxi_zones["LocationID"].astype("int64")
assert len(taxi_zones) == 263 and taxi_zones["LocationID"].is_unique
zone_info = taxi_zones[["LocationID", "borough", "zone"]].rename(columns={"borough": "zone_borough", "zone": "zone_name"})

station_points = gpd.GeoDataFrame(
    stations[["station_id", "station_complex_id"]],
    geometry=gpd.points_from_xy(stations["lon"], stations["lat"]), crs="EPSG:4326",
).to_crs(taxi_zones.crs)
station_zones = gpd.sjoin(station_points, taxi_zones[["LocationID", "geometry"]], how="left", predicate="within")

assert station_zones["LocationID"].notna().all(), "A station fell outside every taxi zone"
assert station_zones["station_id"].is_unique, "A station matched several taxi zones"
assert station_zones["LocationID"].ne(1).all(), "A station matched Newark Airport"

station_complex_zone_bridge = (station_zones.astype({"LocationID": "int64"})[["station_complex_id", "LocationID"]]
                               .drop_duplicates()
                               .merge(zone_info, on="LocationID", how="left", validate="many_to_one")
                               .sort_values(["station_complex_id", "LocationID"]).reset_index(drop=True))
subway_zones = (station_complex_zone_bridge[["LocationID", "zone_borough", "zone_name"]].drop_duplicates()
                .sort_values("LocationID").reset_index(drop=True))
print("Bridge rows:", len(station_complex_zone_bridge), "| Subway-access zones:", len(subway_zones))

### How many riders normally use each station?

Losing service at a busy complex may matter differently from losing it at a quiet one. I'll estimate normal hourly ridership from quarterly 28-day samples. Those profiles give me weights for possible exposure; they cannot tell me how many people were actually waiting during an incident.


In [ ]:
# Normal subway riders per station complex and local hour (MTA Subway Hourly Ridership, NY Open Data).
# A typical level needs only a sample: 28 consecutive days from the middle month of each quarter.
# Each day is one small request, cached on its own, so an interrupted download resumes where it stopped.
RIDERSHIP_SAMPLE_DAYS = 28
SOCRATA_APP_TOKEN = None  # optional free token from data.ny.gov; raises the request limits


def ridership_dataset(day):
    return [dataset for start, dataset in RIDERSHIP_DATASETS if day >= pd.Timestamp(start)][-1]


def ridership_day(day):
    path = RIDERSHIP_DIR / "daily" / f"{day:%Y-%m-%d}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    if not ALLOW_DOWNLOADS:
        raise FileNotFoundError(f"Missing ridership cache: {path}")
    headers = {"X-App-Token": SOCRATA_APP_TOKEN} if SOCRATA_APP_TOKEN else {}
    response = session.get(f"https://data.ny.gov/resource/{ridership_dataset(day)}.json", timeout=180, headers=headers, params={
        "$select": "station_complex_id, transit_timestamp, sum(ridership) AS riders",
        "$where": f"transit_timestamp >= '{day:%Y-%m-%dT00:00:00}' "
                  f"AND transit_timestamp < '{day + pd.Timedelta(days=1):%Y-%m-%dT00:00:00}'",
        "$group": "station_complex_id, transit_timestamp",
        "$limit": 50_000,
    })
    response.raise_for_status()
    frame = pd.DataFrame(response.json(), columns=["station_complex_id", "transit_timestamp", "riders"])
    timestamps = pd.to_datetime(frame["transit_timestamp"])
    frame = pd.DataFrame({"station_complex_id": frame["station_complex_id"].astype(str),
                          "day": timestamps.dt.normalize(), "hour": timestamps.dt.hour.astype(int),
                          "riders": pd.to_numeric(frame["riders"])})
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(path, index=False)
    return frame


def sample_days(quarter):
    """28 consecutive days starting on the first Monday of the quarter's middle month."""
    middle = quarter.start_time + pd.offsets.MonthBegin(1)
    first_monday = middle + pd.Timedelta(days=(7 - middle.dayofweek) % 7)
    return pd.date_range(first_monday, periods=RIDERSHIP_SAMPLE_DAYS, freq="D")


def day_type(dates):
    """Weekday / Saturday / Sunday, with federal holidays as Sunday."""
    dates = pd.Series(pd.to_datetime(dates))
    kind = np.select([dates.dt.dayofweek.lt(5), dates.dt.dayofweek.eq(5)], ["Weekday", "Saturday"], "Sunday")
    return pd.Series(np.where(dates.dt.date.isin(holidays), "Sunday", kind), index=dates.index)


if RUN_MODE == "pilot":
    first_day, last_day = PILOT_DAY.tz_localize(None), PILOT_DAY.tz_localize(None) + pd.Timedelta(days=2)
else:
    first_day, last_day = STUDY_START.tz_localize(None), STUDY_END.tz_localize(None) - pd.Timedelta(days=1)
ridership_quarters = pd.period_range(first_day, last_day, freq="Q")
ridership_days = [d for q in ridership_quarters for d in sample_days(q)]

ridership_frames, failed_days = [], []
for i, day in enumerate(ridership_days, start=1):
    try:
        ridership_frames.append(ridership_day(day))
    except Exception as error:  # a failed day is retried on the next run
        failed_days.append((f"{day:%Y-%m-%d}", f"{type(error).__name__}"))
    if i % 28 == 0:
        print(f"Ridership days: {i} / {len(ridership_days)}")
if failed_days:
    warnings.warn(f"{len(failed_days)} ridership day(s) failed and will be retried on the next run: {failed_days[:5]}")
hourly_ridership = pd.concat(ridership_frames, ignore_index=True)

# Typical riders: mean over days of the same quarter and day type (hours without a row count as zero riders)
hourly_ridership["quarter"] = hourly_ridership["day"].dt.to_period("Q").astype(str)
hourly_ridership["day_type"] = day_type(hourly_ridership["day"]).to_numpy()
days_per_group = hourly_ridership.groupby(["quarter", "day_type"])["day"].nunique().rename("days")
normal_riders = (hourly_ridership.groupby(["station_complex_id", "quarter", "day_type", "hour"])["riders"].sum()
                 .reset_index().merge(days_per_group.reset_index(), on=["quarter", "day_type"]))
normal_riders["normal_riders"] = normal_riders["riders"] / normal_riders["days"]
normal_riders = normal_riders.drop(columns=["riders", "days"])
normal_riders_lookup = normal_riders.set_index(["station_complex_id", "quarter", "day_type", "hour"])["normal_riders"]

matched = stations["station_complex_id"].drop_duplicates().isin(normal_riders["station_complex_id"])
print(f"Ridership sample days: {hourly_ridership['day'].nunique()} in {len(ridership_quarters)} quarter(s) | "
      f"station-list complexes with ridership: {matched.mean():.1%}")
missing_ridership = stations.loc[~stations["station_complex_id"].isin(normal_riders["station_complex_id"])]
if len(missing_ridership):
    display(missing_ridership.drop_duplicates("station_complex_id")
            [["station_complex_id", "station_name", "station_borough", "daytime_routes"]])


def riders_for(complex_ids, times_utc):
    """Normal hourly riders at each complex for the New York local hour of each UTC time."""
    local = to_local_naive(pd.Series(times_utc).reset_index(drop=True))
    keys = pd.MultiIndex.from_arrays([pd.Series(complex_ids).astype(str).reset_index(drop=True),
                                      local.dt.to_period("Q").astype(str), day_type(local).reset_index(drop=True),
                                      local.dt.hour])
    return normal_riders_lookup.reindex(keys).fillna(0).to_numpy()

Ridership days: 28 / 224
Ridership days: 56 / 224
Ridership days: 84 / 224
Ridership days: 112 / 224
Ridership days: 140 / 224
Ridership days: 168 / 224
Ridership days: 196 / 224
Ridership days: 224 / 224
Ridership sample days: 224 in 8 quarter(s) | station-list complexes with ridership: 95.7%


,station_complex_id,station_name,station_borough,daytime_routes
3,515,Annadale,Staten Island,"(SIR,)"
57,507,Dongan Hills,Staten Island,"(SIR,)"
94,503,Stapleton,Staten Island,"(SIR,)"
131,516,Huguenot,Staten Island,"(SIR,)"
158,511,Oakwood Heights,Staten Island,"(SIR,)"
176,506,Old Town,Staten Island,"(SIR,)"
200,523,Arthur Kill,Staten Island,"(SIR,)"
210,510,New Dorp,Staten Island,"(SIR,)"
274,512,Bay Terrace,Staten Island,"(SIR,)"
290,517,Prince's Bay,Staten Island,"(SIR,)"


## 4. Where do the train records show service?

The train archive helps answer where trains appeared to run. It is reconstructed from the realtime feed, though, so the stop times began as estimates. I'll use it as evidence of service patterns without treating every retained stop as a verified physical call.


### Reducing the archives

I reduce the daily archives to the fields this measurement needs. When a trip appears more than once, I keep its latest recorded version and log the duplication. That leaves smaller files for repeated comparisons.


In [ ]:
# Days of train data needed in this run
if RUN_MODE == "pilot":
    needed = {shift_local_days(PILOT_DAY, d) for d in (-1, 0, 1)}
    for k in range(1, BASELINE_SEARCH_WEEKS + 1):
        needed |= {shift_local_days(PILOT_DAY, d - 7 * k) for d in (-1, 0, 1)}
    train_days = pd.DatetimeIndex(sorted(needed))
elif RUN_MODE == "full":
    train_days = local_dates(shift_local_days(TRAIN_START, -1), shift_local_days(TRAIN_END, 1))
else:
    raise ValueError("RUN_MODE must be 'pilot' or 'full'")
print("Train archive days in this run:", len(train_days), f"({train_days.min():%Y-%m-%d} to {train_days.max():%Y-%m-%d})")


def validate_train_archive(path):
    with tarfile.open(path, "r:xz") as archive:
        names = [m.name for m in archive.getmembers() if m.isfile()]
    for table in ("trips", "stop_times"):
        assert sum(n.endswith(f"_{table}.csv") for n in names) == 1, f"{path.name} lacks {table}"


def reduce_train_day(day):
    """Reducing one archive to unique trip x stop records with exact times (independent of station mapping)."""
    label = f"{day:%Y-%m-%d}"
    reduced_path = TRAIN_REDUCED_DIR / f"{label}.parquet"
    if reduced_path.exists():
        return {"day": label, "status": "cached"}

    archive_path = BRONZE_TRAIN_DIR / f"subwaydatanyc_{label}_csv.tar.xz"
    try:
        fetch_cached_file(TRAIN_ARCHIVE_URL.format(day=label), archive_path, validator=validate_train_archive)
    except NotAtSource:
        return {"day": label, "status": "not_at_source"}
    except Exception as error:
        return {"day": label, "status": "failed", "error": f"{type(error).__name__}: {str(error)[:200]}"}

    tables = {}
    with tarfile.open(archive_path, "r:xz") as archive:
        for name in ("trips", "stop_times"):
            member = next(m for m in archive.getmembers() if m.isfile() and m.name.endswith(f"_{name}.csv"))
            with archive.extractfile(member) as source:
                tables[name] = pd.read_csv(source, dtype="string", keep_default_na=False)

    # Some archives list a trip_uid more than once; keeping its most recently observed row
    trips = tables["trips"].copy()
    duplicated = trips["trip_uid"].duplicated(keep=False)
    duplicate_trip_rows = int(duplicated.sum())
    conflicting_route_trips = int(trips.loc[duplicated].groupby("trip_uid")["route_id"].nunique().gt(1).sum())
    if "last_observed" in trips.columns:
        trips["_order"] = pd.to_numeric(trips["last_observed"].replace("", pd.NA), errors="coerce")
        trips = trips.sort_values("_order", na_position="first", kind="stable")
    trips = trips.drop_duplicates("trip_uid", keep="last")

    stops = tables["stop_times"].merge(trips[["trip_uid", "route_id"]], on="trip_uid", how="inner", validate="many_to_one")
    arrival = pd.to_numeric(stops["arrival_time"].replace("", pd.NA), errors="coerce")
    departure = pd.to_numeric(stops["departure_time"].replace("", pd.NA), errors="coerce")
    stops["time_s"] = arrival.fillna(departure)
    stops = stops.dropna(subset=["time_s"])
    stops["time_s"] = stops["time_s"].astype("int64")
    stops["direction"] = stops["stop_id"].str[-1]
    stops["parent"] = stops["stop_id"].str[:-1]
    # "marked_past" means the realtime feed marked the stop as passed; it is not a verified station call
    stops["confirmed"] = stops["marked_past"].ne("") if "marked_past" in stops.columns else pd.NA

    reduced = (stops.sort_values("time_s")
               .drop_duplicates(["trip_uid", "stop_id"], keep="first")
               [["trip_uid", "route_id", "parent", "direction", "time_s", "confirmed"]])
    reduced.to_parquet(reduced_path, index=False)
    if not KEEP_RAW_TRAIN_ARCHIVES:
        archive_path.unlink(missing_ok=True)
    return {"day": label, "status": "reduced", "stop_rows": len(reduced),
            "duplicate_trip_rows": duplicate_trip_rows, "conflicting_route_trips": conflicting_route_trips}


from concurrent.futures import ThreadPoolExecutor

# Days are independent and write their own files, so they can be reduced in parallel threads
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    reduction_log = pd.DataFrame(list(pool.map(reduce_train_day, train_days)))
display(reduction_log["status"].value_counts().to_frame("days"))
display(reduction_log.loc[reduction_log["status"].isin(["not_at_source", "failed"])])
if "duplicate_trip_rows" in reduction_log:
    display(reduction_log.loc[reduction_log["duplicate_trip_rows"].fillna(0).gt(0),
                              ["day", "duplicate_trip_rows", "conflicting_route_trips"]])

### How do I count trains across midnight?

A train that starts before midnight can stop after midnight. I'll combine neighboring daily archives so those stops do not disappear at a file boundary. Calendar-day handling also avoids assuming that every New York day has exactly 24 hours.


In [ ]:
# Counts depend on the station mapping, so they are stored under a fingerprint of it
MAPPING_FINGERPRINT = hashlib.sha256(
    (STATIONS_SHA256 + json.dumps(MANUAL_STOP_PARENT, sort_keys=True) + "calendar-days-v2").encode()).hexdigest()[:12]
TRAIN_COUNTS_DIR = TRAIN_COUNTS_ROOT / MAPPING_FINGERPRINT
TRAIN_COUNTS_DIR.mkdir(parents=True, exist_ok=True)


def build_daily_counts(day):
    """Trains per route x direction x complex x bin for one local date, combining neighboring archives."""
    label = f"{day:%Y-%m-%d}"
    paths = {name: TRAIN_COUNTS_DIR / f"{label}_{name}.parquet" for name in ("counts", "system", "unmapped")}
    sources = {d: TRAIN_REDUCED_DIR / f"{shift_local_days(day, d):%Y-%m-%d}.parquet" for d in (-1, 0, 1)}
    available = {d: p for d, p in sources.items() if p.exists()}
    status = {"day_ny": label, "previous_archive": -1 in available, "next_archive": 1 in available}

    newest_source = max((p.stat().st_mtime for p in available.values()), default=0)
    if all(p.exists() for p in paths.values()) and paths["counts"].stat().st_mtime >= newest_source:
        return {**status, "status": "cached"}
    if 0 not in available:
        return {**status, "status": "missing"}

    start = pd.Timestamp(label, tz=LOCAL_TIMEZONE).tz_convert("UTC")
    end = shift_local_days(pd.Timestamp(label, tz=LOCAL_TIMEZONE), 1).tz_convert("UTC")
    records = pd.concat([pd.read_parquet(p) for p in available.values()], ignore_index=True)
    records["time_utc"] = pd.to_datetime(records["time_s"], unit="s", utc=True)
    records = records.loc[records["time_utc"].ge(start) & records["time_utc"].lt(end)].copy()

    records["parent"] = records["parent"].replace(MANUAL_STOP_PARENT)
    records["station_complex_id"] = records["parent"].map(station_to_complex)
    unmapped = (records.loc[records["station_complex_id"].isna()].groupby(["parent", "route_id"]).trip_uid.nunique()
                .rename("trips").reset_index().assign(day=label))

    records = (records.dropna(subset=["station_complex_id"]).sort_values("time_utc")
               .drop_duplicates(["trip_uid", "direction", "station_complex_id"], keep="first"))
    records["bin_utc"] = records["time_utc"].dt.floor("15min")
    records["feed_group"] = records["route_id"].map(ROUTE_FEED_GROUP).fillna("other")

    counts = (records.groupby(["route_id", "direction", "station_complex_id", "bin_utc"]).trip_uid.nunique()
              .rename("trains").reset_index())
    system = records.groupby(["feed_group", "bin_utc"]).trip_uid.nunique().rename("trips").reset_index()
    counts.to_parquet(paths["counts"], index=False)
    system.to_parquet(paths["system"], index=False)
    unmapped.to_parquet(paths["unmapped"], index=False)
    return {**status, "status": "built"}


count_days = pd.DatetimeIndex([d for d in train_days
                               if shift_local_days(d, -1) in set(train_days) or RUN_MODE == "pilot"])
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    count_log = pd.DataFrame(list(pool.map(build_daily_counts, count_days)))
display(count_log.groupby(["status", "previous_archive", "next_archive"]).size().to_frame("days"))


# Every study day must have a count entry (catches calendar errors such as the daylight-saving bug)
if RUN_MODE == "full":
    expected_days = set(local_dates(STUDY_START, STUDY_END).strftime("%Y-%m-%d"))
    missing_count_dates = sorted(expected_days - set(count_log["day_ny"]))
    assert not missing_count_dates, f"Study days without a daily-count entry: {missing_count_dates}"
    print("Every study day has a daily-count entry.")


@lru_cache(maxsize=128)
def load_counts(label):
    path = TRAIN_COUNTS_DIR / f"{label}_counts.parquet"
    return pd.read_parquet(path) if path.exists() else None


@lru_cache(maxsize=1024)
def load_system(label):
    path = TRAIN_COUNTS_DIR / f"{label}_system.parquet"
    return pd.read_parquet(path) if path.exists() else None


train_available_days = set(count_log.loc[count_log["status"].ne("missing"), "day_ny"])
realtime_route_ids = set(pd.concat([load_counts(d)["route_id"] for d in sorted(train_available_days)]).unique())
print("Realtime route ids:", sorted(realtime_route_ids))

days
status previous_archive next_archive      
cached True             False            1
                        True           822

Every study day has a daily-count entry.
Realtime route ids: ['1', '2', '3', '4', '5', '6', '6X', '7', '7X', 'A', 'B', 'C', 'D', 'E', 'F', 'FS', 'FX', 'G', 'GS', 'H', 'J', 'L', 'M', 'N', 'Q', 'R', 'SI', 'SS', 'W', 'Z']


### Why do some stop IDs have no station?

Some recorded IDs are timing points or other locations outside the passenger-station reference. I'll inspect those unmatched IDs before dropping them. The important question is what the point represents, not simply whether a merge found it.


In [ ]:
# Realtime stops that are not passenger stations in the MTA list
unmapped_stops = pd.concat([pd.read_parquet(TRAIN_COUNTS_DIR / f"{d}_unmapped.parquet")
                            for d in sorted(train_available_days)
                            if (TRAIN_COUNTS_DIR / f"{d}_unmapped.parquet").exists()], ignore_index=True)
if unmapped_stops.empty:
    print("Every realtime stop maps to a listed station.")
else:
    summary = (unmapped_stops.groupby(["parent", "route_id"]).agg(days=("day", "nunique"), trips=("trips", "sum"))
               .reset_index().sort_values("trips", ascending=False))
    # Unmapped means "not in the MTA station list": these are not passenger stations by construction
    assert not summary["parent"].isin(station_to_complex).any()
    print(f"{summary['parent'].nunique()} realtime stop ids are not passenger stations in the MTA list "
          "(timing points, junctions, non-revenue track). Their records are dropped.")
    display(summary.head(30))

### Route names and helper queries

The sources use different names for some of the same routes. I'll normalize express variants and shuttle names, then print unmatched routes for review. A naming mismatch should not look like an absence of service.


In [ ]:
ROUTE_ALIASES = {"6": ("6", "6X"), "7": ("7", "7X"), "F": ("F", "FX"), "SIR": ("SI",), "SI": ("SI",)}
SHUTTLE_BY_TEXT = [("GS", r"\b42 St\b|\bTimes Sq\b|\bGrand Central\b"), ("FS", r"\bFranklin\b"), ("H", r"\bRockaway\b")]


def to_realtime_routes(symbols, text=""):
    """Alert route symbols to realtime route ids; the shuttle is chosen from the text when possible."""
    routes, unmapped = set(), set()
    for symbol in symbols:
        if symbol == "S":
            named = [r for r, pattern in SHUTTLE_BY_TEXT if r in symbols or re.search(pattern, text)]
            candidates = set(named or [r for r, _ in SHUTTLE_BY_TEXT])
        else:
            candidates = set(ROUTE_ALIASES.get(symbol, (symbol,)))
        if not candidates & realtime_route_ids:
            unmapped.add(symbol)
        routes |= candidates & realtime_route_ids
    return tuple(sorted(routes)), tuple(sorted(unmapped))


mapped = [to_realtime_routes(s, t) for s, t in zip(episodes["affected_symbols"], episodes["episode_text"])]
episodes["routes"] = [m[0] for m in mapped]
episodes["unmapped_symbols"] = [m[1] for m in mapped]
episodes["disrupted_routes"] = [to_realtime_routes(s, t)[0] for s, t in zip(episodes["disrupted_symbols"], episodes["episode_text"])]
episodes["alternative_routes"] = [tuple(sorted(set(r) - set(d))) for r, d in zip(episodes["routes"], episodes["disrupted_routes"])]

unmapped_symbol_counts = episodes["unmapped_symbols"].explode().dropna().value_counts()
if len(unmapped_symbol_counts):
    print("Alert route symbols with no realtime route (add to ROUTE_ALIASES if they are real routes):")
    display(unmapped_symbol_counts.to_frame("episodes"))

# Disruption intervals per disrupted route, used to keep disrupted days out of baselines
route_intervals = (episodes[["disrupted_routes", "incident_start", "alert_end"]].explode("disrupted_routes")
                   .dropna(subset=["disrupted_routes"]).groupby("disrupted_routes")
                   .agg(starts=("incident_start", list), ends=("alert_end", list)))
route_intervals = {r: (pd.DatetimeIndex(row.starts), pd.DatetimeIndex(row.ends)) for r, row in route_intervals.iterrows()}


def route_disrupted(routes, start, end):
    for route in routes:
        if route in route_intervals:
            starts, ends = route_intervals[route]
            if ((starts < end) & (ends > start)).any():
                return True
    return False


def window_counts(start, end, routes=None):
    """Counts in [start, end) UTC, or None if any needed day is missing."""
    first = start.tz_convert(LOCAL_TIMEZONE).normalize()
    last = (end - pd.Timedelta(1, unit="ns")).tz_convert(LOCAL_TIMEZONE).normalize()
    frames = []
    for label in {f"{d:%Y-%m-%d}" for d in pd.date_range(first, last, freq="D")}:
        counts = load_counts(label)
        if counts is None:
            return None
        mask = counts["bin_utc"].ge(start) & counts["bin_utc"].lt(end)
        if routes is not None:
            mask &= counts["route_id"].isin(routes)
        frames.append(counts.loc[mask])
    return pd.concat(frames, ignore_index=True)


def window_feed_status(start, end):
    """Feed health per bin and feed group in [start, end) UTC (True when healthy or not judged)."""
    bins = pd.date_range(start, end, freq="15min", inclusive="left")
    local = bins.tz_convert(LOCAL_TIMEZONE)
    days, clocks = local.strftime("%Y-%m-%d"), local.strftime("%H:%M")
    return pd.DataFrame({group: [feed_ok_lookup.get((d, c, group), True) for d, c in zip(days, clocks)]
                         for group in [*FEED_GROUPS, "other"]}, index=bins)


def baseline_candidates(start):
    """Day shifts for baseline windows: same weekday, or earlier Sundays when the episode day is a holiday."""
    day = start.tz_convert(LOCAL_TIMEZONE).normalize().tz_localize(None)
    if day.date() in holidays:
        last_sunday = day - pd.Timedelta(days=(day.dayofweek + 1) % 7 or 7)
        return [(day - (last_sunday - pd.Timedelta(weeks=k))).days for k in range(BASELINE_SEARCH_WEEKS)]
    return [7 * k for k in range(1, BASELINE_SEARCH_WEEKS + 1)]

### How can I tell an outage from a missing feed?

A missing feed can look exactly like a stopped line: no recorded trains. I'll compare activity within feed groups and look for announced information outages. These checks help keep data failures separate from service loss, although the distinction is not always certain.


In [ ]:
# Feed health per feed group: trips seen per bin compared with the same clock time on earlier same weekdays
def local_bins(label):
    start = pd.Timestamp(label, tz=LOCAL_TIMEZONE)
    return pd.date_range(start, shift_local_days(start, 1), freq="15min", inclusive="left").tz_convert("UTC")


@lru_cache(maxsize=4096)
def group_series(label, group):
    system = load_system(label)
    if system is None:
        return None
    bins = local_bins(label)
    series = (system.loc[system["feed_group"].eq(group)].set_index("bin_utc")["trips"]
              .reindex(bins, fill_value=0))
    return series.set_axis(bins.tz_convert(LOCAL_TIMEZONE).strftime("%H:%M")).groupby(level=0).sum()


def history_days(day):
    """Earlier comparison days: same weekday, or earlier Sundays for a holiday."""
    if day.date() in holidays:
        last_sunday = day - pd.Timedelta(days=(day.dayofweek + 1) % 7 or 7)
        return [last_sunday - pd.Timedelta(weeks=k) for k in range(BASELINE_WEEKS)]
    return [day - pd.Timedelta(weeks=k) for k in range(1, BASELINE_WEEKS + 1)]


count_status = count_log.set_index("day_ny")

# Information outages: alerts saying arrival information is inaccurate mean the realtime data for those routes
# (the source of the train records) cannot be trusted during the alert
info_states = timeline.loc[timeline["status_label"].str.contains("information-outage", na=False),
                           ["alert_id", "date", "next_update_time", "affected", "incident_text"]].copy()
info_states["start"] = info_states["date"]
info_states["end"] = info_states["next_update_time"].fillna(info_states["date"] + MAX_STATE_AGE)
info_states["routes"] = [to_realtime_routes({t.strip() for t in str(a).split("|") if t.strip()}, txt)[0]
                         for a, txt in zip(info_states["affected"], info_states["incident_text"])]
info_outage_windows = (info_states.explode("routes").dropna(subset=["routes"])
                       .assign(feed_group=lambda f: f["routes"].map(ROUTE_FEED_GROUP).fillna("other"))
                       [["feed_group", "start", "end", "alert_id"]].drop_duplicates(["feed_group", "alert_id"]))
group_info_windows = {group: (pd.DatetimeIndex(rows["start"]), pd.DatetimeIndex(rows["end"]))
                      for group, rows in info_outage_windows.groupby("feed_group")}
print("Information-outage alert states:", len(info_states),
      "| feed-group windows:", len(info_outage_windows))

# Alert episodes per feed group: a drop in trips while an alert is active is a service change, not a feed outage
group_alert_windows = {}
for route, (starts, ends) in route_intervals.items():
    group = ROUTE_FEED_GROUP.get(route, "other")
    previous = group_alert_windows.get(group, (pd.DatetimeIndex([], tz="UTC"), pd.DatetimeIndex([], tz="UTC")))
    group_alert_windows[group] = (previous[0].append(starts - pd.Timedelta(hours=1)),
                                  previous[1].append(ends + pd.Timedelta(hours=1)))
feed_rows = []
for label in sorted(train_available_days):
    day = pd.Timestamp(label)
    for group in FEED_GROUPS:
        observed = group_series(label, group)
        history = [group_series(f"{d:%Y-%m-%d}", group) for d in history_days(day)]
        history = [h for h in history if h is not None]
        expected = (pd.concat(history, axis=1).reindex(observed.index).fillna(0).median(axis=1)
                    if len(history) >= MIN_BASELINE_DAYS else pd.Series(np.nan, index=observed.index))
        frame = pd.DataFrame({"day_ny": label, "clock": observed.index, "feed_group": group,
                              "trips": observed.to_numpy(), "expected_trips": expected.to_numpy()})
        hour = frame["clock"].str[:2].astype(int)
        starts, ends = group_alert_windows.get(group, (pd.DatetimeIndex([], tz="UTC"), pd.DatetimeIndex([], tz="UTC")))
        bins = local_bins(label)
        nearby = (starts < bins[-1] + BIN) & (ends > bins[0])
        naive = lambda index: index.tz_convert("UTC").tz_localize(None).to_numpy()
        bin_values = naive(bins)[:, None]
        alert_active = ((bin_values >= naive(starts[nearby])[None, :])
                        & (bin_values < naive(ends[nearby])[None, :])).any(axis=1) if nearby.any() else np.zeros(len(bins), bool)
        alert_active = pd.Series(alert_active, index=bins.tz_convert(LOCAL_TIMEZONE).strftime("%H:%M")).groupby(level=0).max()
        alert_active = frame["clock"].map(alert_active).fillna(False).to_numpy()

        info_starts, info_ends = group_info_windows.get(group, (pd.DatetimeIndex([], tz="UTC"), pd.DatetimeIndex([], tz="UTC")))
        info_nearby = (info_starts < bins[-1] + BIN) & (info_ends > bins[0])
        info_active = (((bin_values + np.timedelta64(15, "m") > naive(info_starts[info_nearby])[None, :])
                        & (bin_values < naive(info_ends[info_nearby])[None, :])).any(axis=1)
                       if info_nearby.any() else np.zeros(len(bins), bool))
        info_active = pd.Series(info_active, index=bins.tz_convert(LOCAL_TIMEZONE).strftime("%H:%M")).groupby(level=0).max()
        info_active = frame["clock"].map(info_active).fillna(False).to_numpy()
        frame["status"] = np.select(
            [info_active,
             hour.lt(3) & (not bool(count_status.loc[label, "previous_archive"])),
             hour.ge(21) & (not bool(count_status.loc[label, "next_archive"])),
             frame["expected_trips"].isna() | frame["expected_trips"].lt(FEED_MIN_EXPECTED),
             frame["trips"].lt(FEED_OK_RATIO * frame["expected_trips"]) & alert_active,
             frame["trips"].lt(FEED_OK_RATIO * frame["expected_trips"])],
            ["info_outage", "incomplete", "incomplete", "unjudged", "service_change", "gap"], "ok")
        feed_rows.append(frame)

feed_health = pd.concat(feed_rows, ignore_index=True)
feed_health["feed_ok"] = feed_health["status"].isin(["ok", "unjudged", "service_change"])
feed_ok_lookup = feed_health.set_index(["day_ny", "clock", "feed_group"])["feed_ok"].to_dict()

display(feed_health.groupby(["feed_group", "status"]).size().unstack(fill_value=0))
display(feed_health.loc[feed_health["status"].isin(["gap", "incomplete"])]
        .groupby(["day_ny", "feed_group", "status"]).size().rename("bins").reset_index().head(40))

Information-outage alert states: 94 | feed-group windows: 232


status,gap,incomplete,info_outage,ok,service_change,unjudged
feed_group,,,,,,
1234567S,559,12,942,75152,346,1989
ACE,210,12,264,76432,66,2016
BDFM,292,12,264,76223,193,2016
G,443,12,251,70265,91,7938
JZ,796,12,264,75808,67,2053
L,1198,12,209,75364,68,2149
NQRW,163,12,251,76526,32,2016
SI,145,12,55,25463,60,53265


,day_ny,feed_group,status,bins
0,2024-04-22,G,gap,1
1,2024-04-27,SI,gap,6
2,2024-04-28,SI,gap,1
3,2024-04-30,G,gap,2
4,2024-05-03,G,gap,1
5,2024-05-04,G,gap,8
6,2024-05-04,L,gap,1
7,2024-05-05,G,gap,2
8,2024-05-05,JZ,gap,3
9,2024-05-08,SI,gap,1


## 5. How far did service fall below what usually runs?

Now I can compare an episode with what normally runs at the same place and time. The goal is to find a sustained change, not label every empty 15-minute bin as a disruption.


### What would I expect on an ordinary day?

I look back for comparable days with usable data and no relevant route disruption, taking up to six and requiring at least three. Their median gives the expected train counts. The table below explains how loss, partial service, access loss, and diversion differ.

For example, if I normally expect five trains across two consecutive 15-minute bins and I see none, that can meet the weekday daytime rule for route loss. If another route still serves the station in that direction, I record the loss of the affected route. Riders still had some subway access.

I also check what was already abnormal in the first 45 minutes. Existing work should not suddenly be blamed on the new incident. A pre-alert gap needs to continue into the alerted problem before I attribute it to that episode.

| Flag | Rule | What it captures |
|---|---|---|
| Route loss | A run of empty bins in which enough trains were expected | The disrupted route stopped serving the station |
| Partial loss | A run of bins at or below half of the expected trains | Service thinned out (typical of severe delays) |
| Access loss | Route loss, and no train of any route served the station in that direction | Riders there had no train at all |
| Diversion | Trains where the route does not normally run in this window | The rerouted corridor |


In [ ]:
def run_ids(frame, group_cols, condition):
    """An id per run of consecutive equal values of `condition` within each series (frame sorted by series, time)."""
    series_id = frame.groupby(group_cols, sort=False).ngroup()
    change = condition.ne(condition.shift()) | series_id.ne(series_id.shift())
    return change.cumsum()


def flag_empty_runs(frame, group_cols, observed, expected, min_expected):
    """Bins in runs of consecutive empty bins (healthy feed) whose expected trains add up to `min_expected`."""
    empty = frame[observed].eq(0) & frame["feed_ok"]
    runs = run_ids(frame, group_cols, empty)
    run_expected = frame[expected].groupby(runs).transform("sum")
    run_length = runs.groupby(runs).transform("size")
    return empty & run_expected.ge(min_expected) & run_length.ge(LOSS_MIN_RUN_BINS)


def flag_partial_runs(frame, group_cols, observed, expected):
    """Bins in runs of consecutive low bins (at most PARTIAL_LOSS_SHARE of expected) that add up to a clear deficit."""
    low = (frame[observed].le(PARTIAL_LOSS_SHARE * frame[expected]) & frame[expected].ge(0.5) & frame["feed_ok"])
    runs = run_ids(frame, group_cols, low)
    run_expected = frame[expected].groupby(runs).transform("sum")
    run_observed = frame[observed].groupby(runs).transform("sum")
    run_length = runs.groupby(runs).transform("size")
    return (low & run_expected.ge(PARTIAL_LOSS_MIN_EXPECTED)
            & run_observed.le(PARTIAL_LOSS_SHARE * run_expected) & run_length.ge(LOSS_MIN_RUN_BINS))


FLAGS = ["lost", "partial_lost", "access_lost", "diverted"]
KEY = ["route_id", "direction", "station_complex_id"]


def measure_episode(ep, complete_min_expected=None):
    """Comparing trains on the episode's routes with baseline days at the same local clock time."""
    complete_min_expected = complete_min_expected or COMPLETE_LOSS_MIN_EXPECTED[broad_period(ep.period)]
    routes = tuple(ep.routes)
    info = {"episode_key": ep.episode_key}
    if not routes:
        return {**info, "measure_status": "no_routes"}, None

    start = (ep.incident_start - PRE_WINDOW).floor("15min")
    end = (ep.alert_end + POST_WINDOW).ceil("15min")
    info["window_truncated"] = bool(end - start > MAX_MEASURE_WINDOW)
    end = min(end, start + MAX_MEASURE_WINDOW)
    bins = pd.date_range(start, end, freq="15min", inclusive="left")
    start_wall = start.tz_convert(LOCAL_TIMEZONE).tz_localize(None)
    end_wall = end.tz_convert(LOCAL_TIMEZONE).tz_localize(None)
    bin_offsets = ((bins.tz_convert(LOCAL_TIMEZONE).tz_localize(None) - start_wall) / BIN).astype(int)

    observed_all = window_counts(start, end)
    if observed_all is None:
        return {**info, "measure_status": "no_train_data"}, None

    # Baseline days: same local clock time on earlier comparable days, skipping unusable ones
    route_groups = sorted({ROUTE_FEED_GROUP.get(r, "other") for r in routes})
    exclusion_routes = tuple(ep.disrupted_routes) or routes
    rejected = {"holiday": 0, "own_disruption": 0, "feed": 0, "missing_data": 0}
    baselines = []
    for shift_days in baseline_candidates(start):
        b_start_wall = start_wall - pd.Timedelta(days=shift_days)
        b_end_wall = end_wall - pd.Timedelta(days=shift_days)
        b_start, b_end = localize_wall_clock(b_start_wall), localize_wall_clock(b_end_wall)
        if b_start_wall.date() in holidays:
            rejected["holiday"] += 1
            continue
        if route_disrupted(exclusion_routes, b_start - BASELINE_EXCLUSION_BUFFER, b_end + BASELINE_EXCLUSION_BUFFER):
            rejected["own_disruption"] += 1
            continue
        if not window_feed_status(b_start, b_end)[route_groups].all().all():
            rejected["feed"] += 1
            continue
        counts = window_counts(b_start, b_end)
        if counts is None:
            rejected["missing_data"] += 1
            continue
        offsets = ((to_local_naive(counts["bin_utc"]) - b_start_wall) / BIN).astype(int)
        baselines.append(counts.assign(offset=offsets, baseline=shift_days))
        if len(baselines) == BASELINE_WEEKS:
            break
    info["baseline_days"] = len(baselines)
    info.update({f"baseline_rejected_{reason}": n for reason, n in rejected.items()})
    if len(baselines) < MIN_BASELINE_DAYS:
        return {**info, "measure_status": "insufficient_baseline"}, None

    base_all = pd.concat(baselines, ignore_index=True)
    observed = observed_all.loc[observed_all["route_id"].isin(routes)]
    base = base_all.loc[base_all["route_id"].isin(routes)]
    baseline_ids = sorted(base_all["baseline"].unique())

    # Grid: every series x every bin of the window
    keys = pd.concat([observed[KEY], base[KEY]]).drop_duplicates().reset_index(drop=True)
    n_bins = len(bins)
    grid = keys.loc[keys.index.repeat(n_bins)].reset_index(drop=True)
    grid["bin_utc"] = pd.to_datetime(np.tile(bins.tz_localize(None).to_numpy(), len(keys))).tz_localize("UTC")
    grid["offset"] = np.tile(bin_offsets, len(keys))
    grid = grid.merge(observed[KEY + ["bin_utc", "trains"]], on=KEY + ["bin_utc"], how="left")
    grid["trains"] = grid["trains"].fillna(0)

    base_wide = (base.pivot_table(index=KEY + ["offset"], columns="baseline", values="trains", aggfunc="sum")
                 .reindex(columns=baseline_ids))
    grid = grid.merge(base_wide, left_on=KEY + ["offset"], right_index=True, how="left")
    grid[baseline_ids] = grid[baseline_ids].fillna(0)
    grid["expected"] = grid[baseline_ids].median(axis=1)
    served_share = (grid.groupby(KEY)[baseline_ids].transform("max") > 0).mean(axis=1)
    grid["normally_served"] = served_share.ge(NORMALLY_SERVED_SHARE)
    grid = grid.drop(columns=baseline_ids).sort_values(KEY + ["bin_utc"]).reset_index(drop=True)

    # Feed health of each route's feed group
    feed = window_feed_status(start, end)
    grid["feed_group"] = grid["route_id"].map(ROUTE_FEED_GROUP).fillna("other")
    feed_long = feed.stack().rename("feed_ok").rename_axis(["bin_utc", "feed_group"]).reset_index()
    grid = grid.merge(feed_long, on=["bin_utc", "feed_group"], how="left")
    grid["feed_ok"] = grid["feed_ok"].fillna(True).astype(bool)
    grid = grid.sort_values(KEY + ["bin_utc"]).reset_index(drop=True)

    # Route loss (complete and partial)
    grid["lost"] = flag_empty_runs(grid, KEY, "trains", "expected", complete_min_expected)
    grid["partial_lost"] = flag_partial_runs(grid, KEY, "trains", "expected") & ~grid["lost"]

    # Access loss: no train of any route at the complex (in any direction unless configured otherwise)
    access_cols = ["station_complex_id"] if ACCESS_IGNORES_DIRECTION else ["direction", "station_complex_id"]
    access_observed = observed_all.groupby(access_cols + ["bin_utc"]).trains.sum().rename("all_route_trains")
    access_expected = (base_all.groupby(access_cols + ["offset", "baseline"]).trains.sum()
                       .unstack("baseline").reindex(columns=baseline_ids).fillna(0).median(axis=1)
                       .rename("all_route_expected"))
    access = (grid[access_cols + ["bin_utc", "offset"]].drop_duplicates()
              .merge(access_observed, left_on=access_cols + ["bin_utc"], right_index=True, how="left")
              .merge(access_expected, left_on=access_cols + ["offset"], right_index=True, how="left")
              .fillna({"all_route_trains": 0, "all_route_expected": 0})
              .sort_values(access_cols + ["bin_utc"]).reset_index(drop=True))
    access["feed_ok"] = access["bin_utc"].map(feed.all(axis=1)).fillna(True).astype(bool)
    access["access_empty"] = flag_empty_runs(access, access_cols, "all_route_trains", "all_route_expected",
                                             complete_min_expected)
    grid = grid.merge(access[access_cols + ["bin_utc", "all_route_trains", "all_route_expected", "access_empty"]],
                      on=access_cols + ["bin_utc"], how="left", validate="many_to_one")
    grid["access_lost"] = grid["lost"] & grid["access_empty"].fillna(False).astype(bool)

    # Diversion: trains where the route does not normally run in this window
    grid["diverted"] = grid["trains"].ge(1) & ~grid["normally_served"] & grid["feed_ok"]
    diverted_trains_in_window = grid["trains"].where(grid["diverted"], 0).groupby([grid[k] for k in KEY]).transform("sum")
    grid["diverted"] = grid["diverted"] & diverted_trains_in_window.ge(DIVERSION_MIN_TRAINS[broad_period(ep.period)])

    # Background, judged with a fixed rule: series already abnormal in the reference period are not attributed
    # to this episode until they first return to normal service
    grid = grid.sort_values(KEY + ["bin_utc"]).reset_index(drop=True)
    in_reference = grid["bin_utc"] < start + REFERENCE_PERIOD
    background_flag = flag_empty_runs(grid, KEY, "trains", "expected", BACKGROUND_MIN_EXPECTED) | grid["diverted"]
    background_keys = grid.loc[in_reference & background_flag, KEY].drop_duplicates()
    grid = grid.merge(background_keys.assign(background=True), on=KEY, how="left")
    grid["background"] = grid["background"].eq(True)
    in_reference = grid["bin_utc"] < start + REFERENCE_PERIOD
    background_flag = background_flag.to_numpy()
    recovered_bin = (grid.assign(normal=~background_flag & grid["trains"].ge(1) & ~in_reference)
                     .loc[lambda f: f["background"] & f["normal"]].groupby(KEY)["bin_utc"].min()
                     .rename("recovered_bin_utc"))
    grid = grid.merge(recovered_bin, left_on=KEY, right_index=True, how="left")
    allowed = ~grid["background"] | (grid["bin_utc"] > grid["recovered_bin_utc"])
    grid["background_loss"] = grid["background"] & ~allowed & background_flag & ~grid["diverted"]
    for flag in FLAGS:
        grid[f"{flag}_raw"] = grid[flag]
        grid[flag] = grid[flag] & allowed & ~in_reference
    # Rows whose train counts can describe this episode's service level (no background, reference, or feed problem)
    grid["attributable"] = allowed & ~in_reference & grid["feed_ok"]

    grid["phase"] = np.select([grid["bin_utc"] + BIN <= ep.onset_time, grid["bin_utc"] >= ep.alert_end],
                              ["pre_alert", "post_alert"], "during_alert")
    grid["episode_key"] = ep.episode_key

    # Pre-alert flags count only when their run reaches to within PRE_ALERT_CONNECT of onset
    grid = grid.sort_values(KEY + ["bin_utc"]).reset_index(drop=True)
    connect_limit = ep.onset_time - PRE_ALERT_CONNECT
    unconnected_series = set()
    for flag in ["lost", "partial_lost", "diverted"]:
        runs = run_ids(grid, KEY, grid[flag])
        run_end = grid["bin_utc"].groupby(runs).transform("max") + BIN
        unconnected = grid[flag] & grid["phase"].eq("pre_alert") & (run_end < connect_limit)
        unconnected_series |= set(map(tuple, grid.loc[unconnected, KEY].drop_duplicates().to_numpy()))
        grid[f"{flag}_unconnected"] = unconnected
        grid[flag] = grid[flag] & ~unconnected
    grid["access_lost"] = grid["access_lost"] & grid["lost"]

    # Summary
    info.update({"measure_status": "measured", "window_start": start, "window_end": end,
                 "series": len(keys), "background_series": len(background_keys),
                 "feed_problem_bins": int((~feed[route_groups].all(axis=1)).sum()),
                 "unconnected_pre_alert_series": len(unconnected_series)})
    for flag in FLAGS:
        for phase in ["pre_alert", "during_alert", "post_alert"]:
            rows = grid.loc[grid[flag] & grid["phase"].eq(phase)]
            info[f"{flag}_complexes_{phase}"] = rows["station_complex_id"].nunique()
        info[f"{flag}_series_during_alert"] = len(grid.loc[grid[flag] & grid["phase"].eq("during_alert"), KEY].drop_duplicates())
    flagged = grid.loc[grid[FLAGS].any(axis=1)]
    info["first_flag_utc"] = flagged["bin_utc"].min() if len(flagged) else pd.NaT
    info["last_flag_utc"] = flagged["bin_utc"].max() + BIN if len(flagged) else pd.NaT
    first_allowed = start + REFERENCE_PERIOD
    pre_losses = grid.loc[(grid["lost"] | grid["partial_lost"]) & grid["phase"].eq("pre_alert")]
    info["pre_alert_loss_starts_at_reference_end"] = bool(len(pre_losses) and pre_losses["bin_utc"].min() == first_allowed)
    diverted_during = grid.loc[grid["diverted"] & grid["phase"].eq("during_alert")]
    info["diverted_trips_estimate"] = (int(diverted_during.groupby("station_complex_id")["trains"].sum().max())
                                       if len(diverted_during) else 0)
    return info, grid

### Can the rule pick up the N reroute I checked by hand?

The June 30 N reroute is a useful worked example because I have already examined it train by train. I expect loss on the normal route and diverted trains on alternative corridors. I'll see whether the rule recovers that pattern before relying on it across thousands of episodes.


In [ ]:
SEA_BEACH_STATIONS = ["N05", "N06", "N07", "N08", "N09", "N10"]
BRIGHTON_STATIONS = ["D26", "D27", "D28", "D29", "D30", "D31", "D32", "D33", "D34", "D35", "D37", "D38", "D39", "D40", "D41", "D42"]


def inspect_episode(grid, routes=None, direction=None):
    """Station x time labels for flagged series of a measured grid."""
    view = grid.copy()
    if routes is not None:
        view = view.loc[view["route_id"].isin(routes)]
    if direction is not None:
        view = view.loc[view["direction"].eq(direction)]
    view["station"] = (view["route_id"] + " " + view["direction"] + " | "
                       + view["station_complex_id"].map(complex_names).fillna("") + " (" + view["station_complex_id"] + ")")
    view["clock"] = view["bin_utc"].dt.tz_convert(LOCAL_TIMEZONE).dt.strftime("%H:%M")
    view["mark"] = np.select(
        [view["access_lost"], view["lost"], view["partial_lost"], view["diverted"],
         view["background"] & (view["lost_raw"] | view["diverted_raw"] | view["background_loss"])],
        ["NO_ACCESS", "ROUTE_LOST", "PARTIAL", "DIV", "bkg"], "")
    stations_with_flags = view.loc[view[FLAGS].any(axis=1), "station"].unique()
    clocks = view.drop_duplicates("bin_utc").set_index("bin_utc")["clock"]
    table = (view.loc[view["station"].isin(stations_with_flags)]
             .pivot_table(index="station", columns="bin_utc", values="mark", aggfunc="first").rename(columns=clocks))
    display(table)


# Worked example: the June 30, 2024 N reroute
pilot_episodes = episodes.loc[episodes["event_id"].eq(PILOT_EVENT_ID)]
display(pilot_episodes[["episode_key", "onset_time", "alert_end", "observed_effects", "cause",
                        "routes", "disrupted_routes", "alternative_routes"]]
        .assign(onset_ny=lambda f: f["onset_time"].dt.tz_convert(LOCAL_TIMEZONE),
                alert_end_ny=lambda f: f["alert_end"].dt.tz_convert(LOCAL_TIMEZONE)))

pilot_info, pilot_grid = measure_episode(pilot_episodes.iloc[0])
display(pd.Series(pilot_info, name="value").to_frame())

if pilot_grid is not None:
    inspect_episode(pilot_grid, routes=["N"], direction="N")
    print("Background series by route:")
    display(pilot_grid.loc[pilot_grid["background"]].drop_duplicates(KEY)
            .assign(station=lambda f: f["station_complex_id"].map(complex_names))
            .groupby("route_id")["station"].agg(lambda n: ", ".join(sorted(set(n)))).to_frame("background_stations"))

    # Checks that must hold for this hand-verified incident
    during = pilot_grid["phase"].eq("during_alert")
    sea_beach = {station_to_complex[s] for s in SEA_BEACH_STATIONS if s in station_to_complex}
    brighton = {station_to_complex[s] for s in BRIGHTON_STATIONS if s in station_to_complex}
    n_rows = pilot_grid["route_id"].eq("N")
    fifty_ninth = station_to_complex.get("R41")
    checks = {
        "Sea Beach station loses all access during the reroute":
            bool((pilot_grid["access_lost"] & during & n_rows & pilot_grid["station_complex_id"].isin(sea_beach)).any()),
        "N trains diverted to a Brighton-line station":
            bool((pilot_grid["diverted"] & n_rows & pilot_grid["station_complex_id"].isin(brighton)).any()),
        "59 St loses the N for longer than it loses all service":
            int((pilot_grid["lost"] & n_rows & pilot_grid["station_complex_id"].eq(fifty_ninth)).sum())
            > int((pilot_grid["access_lost"] & n_rows & pilot_grid["station_complex_id"].eq(fifty_ninth)).sum()),
    }
    display(pd.Series(checks, name="passed").to_frame())
    assert all(checks.values()), "The worked example no longer matches the hand-verified result"

,episode_key,onset_time,alert_end,observed_effects,cause,routes,disrupted_routes,alternative_routes,onset_ny,alert_end_ny
981,158714-1,2024-07-01 02:18:00+00:00,2024-07-01 03:16:00+00:00,"(reroute,)",switch_problem,"(D, N, Q, R)","(N, R)","(D, Q)",2024-06-30 22:18:00-04:00,2024-06-30 23:16:00-04:00
982,158714-2,2024-07-01 03:32:00+00:00,2024-07-01 04:11:00+00:00,"(severe_delay,)",switch_problem,"(D, N, R)","(D, N, R)",(),2024-06-30 23:32:00-04:00,2024-07-01 00:11:00-04:00


,value
episode_key,158714-1
window_truncated,False
baseline_days,6
baseline_rejected_holiday,0
baseline_rejected_own_disruption,4
baseline_rejected_feed,0
baseline_rejected_missing_data,0
measure_status,measured
window_start,2024-06-30 23:30:00+00:00
window_end,2024-07-01 04:30:00+00:00


bin_utc,19:30,19:45,20:00,20:15,20:30,20:45,21:00,21:15,21:30,21:45,22:00,22:15,22:30,22:45,23:00,23:15,23:30,23:45,00:00,00:15
station,,,,,,,,,,,,,,,,,,,,
N N | 14 St-Union Sq (602),,,,,,,,,,,,ROUTE_LOST,ROUTE_LOST,ROUTE_LOST,,,,,,
N N | 18 Av (66),,,,,,,,,,,,,DIV,DIV,,DIV,DIV,,,
N N | 18 Av (74),,,,,,,,,,,NO_ACCESS,NO_ACCESS,NO_ACCESS,,,,,,,
N N | 20 Av (67),,,,,,,,,,,,,DIV,,,DIV,DIV,,,
N N | 20 Av (75),,,,,,,,,,,NO_ACCESS,NO_ACCESS,NO_ACCESS,,,,,,,
N N | 23 St (14),,,,,,,,,,,,ROUTE_LOST,ROUTE_LOST,ROUTE_LOST,,,,,,
N N | 25 Av (69),,,,,,,,,,,,DIV,DIV,,DIV,,DIV,,,
N N | 28 St (13),,,,,,,,,,,,ROUTE_LOST,ROUTE_LOST,ROUTE_LOST,,,,,,
N N | 30 Av (3),,,,,,,,,,,,,NO_ACCESS,NO_ACCESS,NO_ACCESS,NO_ACCESS,,,,


Background series by route:


,background_stations
route_id,
D,"103 St, 116 St, 125 St, 135 St, 145 St, 161 St-Yankee Stadium, 167 St, 170 St, 174-175 Sts, 182-183 Sts, 25 Av, 25 St, 34 St-Herald Sq, 4 Av-9 St,..."
N,"25 St, 4 Av-9 St, 45 St, 53 St, Borough Hall / Court St, Chambers St / Cortlandt St / Park Place / World Trade Center, City Hall, DeKalb Av, Jay S..."
Q,"Avenue H, Avenue J, Avenue M, Avenue U, Beverley Rd, Cortelyou Rd, Neck Rd, Parkside Av"
R,"45 St, 53 St, 59 St, 77 St, 86 St, 9 Av, Bay Ridge Av, Bay Ridge-95 St, Borough Hall / Court St, Chambers St / Cortlandt St / Park Place / World T..."


,passed
Sea Beach station loses all access during the reroute,True
N trains diverted to a Brighton-line station,True
59 St loses the N for longer than it loses all service,True


### Do rerouted trains keep predicted stops they never made?

There is a potential problem with prediction-based archives: a rerouted train might retain stops it never actually served. I'll look for that in the N example. The empty result is reassuring here, but the feed's `marked_past` label is still not independent proof of a station call.


In [ ]:
# Predicted stops: did any rerouted N trip keep Sea Beach stop records while it was rerouted?
# ("marked_past" only means the realtime feed marked the stop as passed, not a verified station call)
pilot_onset = pilot_episodes.iloc[0]["onset_time"]
pilot_end = pilot_episodes.iloc[0]["alert_end"]
reduced = pd.concat([pd.read_parquet(TRAIN_REDUCED_DIR / f"{shift_local_days(PILOT_DAY, d):%Y-%m-%d}.parquet")
                     for d in (-1, 0, 1)
                     if (TRAIN_REDUCED_DIR / f"{shift_local_days(PILOT_DAY, d):%Y-%m-%d}.parquet").exists()])
reduced["time_utc"] = pd.to_datetime(reduced["time_s"], unit="s", utc=True)
during_reroute = reduced.loc[reduced["route_id"].eq("N") & reduced["direction"].eq("N")
                             & reduced["time_utc"].between(pilot_onset + pd.Timedelta(15, unit="min"), pilot_end)]
rerouted_trips = set(during_reroute.loc[during_reroute["parent"].isin(BRIGHTON_STATIONS + ["B23", "B16", "B12"]), "trip_uid"])
display(during_reroute.loc[during_reroute["trip_uid"].isin(rerouted_trips) & during_reroute["parent"].isin(SEA_BEACH_STATIONS)]
        .rename(columns={"confirmed": "marked_past_in_feed"})
        .groupby("marked_past_in_feed", dropna=False).size().rename("sea_beach_stop_records_on_rerouted_trips").to_frame())

,sea_beach_stop_records_on_rerouted_trips
marked_past_in_feed,


### How long will this take?

Before starting the full run, I time a spread of episodes. This gives me a practical estimate of the work ahead and helps decide whether more complicated execution would be worthwhile.


In [ ]:
# Optional: timing a spread-out sample of episodes to estimate how long measurement and calibration will take
if RUN_TIMING_CHECK:
    import time
    timing_scope = (episodes.loc[episodes["onset_day_ny"].eq(PILOT_DAY.date())] if RUN_MODE == "pilot"
                    else episodes.loc[episodes["within_study_window"]]).sort_values("onset_time")
    sample = timing_scope.iloc[::max(1, len(timing_scope) // 60)].head(60)
    start_time = time.perf_counter()
    for ep in sample.itertuples(index=False):
        measure_episode(ep)
    seconds_per_episode = (time.perf_counter() - start_time) / len(sample)
    print(f"{seconds_per_episode:.2f} s per episode (an upper bound: this sample reuses the cache less than the sorted run)")
    print(f"Measurement: about {seconds_per_episode * len(timing_scope) / 3600:.1f} h for {len(timing_scope)} episodes")
    print(f"Calibration: about {seconds_per_episode * 2 * 5 * 150 / 3600:.1f} h")

## 6. Would this rule flag a disruption on a quiet comparison day?

An empty stretch does not mean the same thing at rush hour and overnight. If ten trains were expected, it is much stronger evidence than if only one was due. I'll compare several thresholds using real episodes and earlier same-time windows with no active alert.

The quiet windows give me a way to judge potential false alarms without choosing the rule from pickup outcomes. Some may still contain unannounced service problems, so I won't treat every placebo flag as a proven error. Later causal bias from mismeasurement can also run in more than one direction.


In [ ]:
# Placebo windows: the same routes and clock times one to three weeks earlier, with no episode on the disrupted routes.
# Unannounced problems or planned work can still occur there, so placebo rates are an upper bound on noise.
COMPLETE_LOSS_GRID = [2.0, 3.0, 4.0, 5.0, 6.0]
PLACEBO_WEEKS = (-1, -2, -3)
CALIBRATION_SAMPLE = 150
BOOTSTRAP_DRAWS = 1000

if RUN_MODE == "pilot":
    scope = episodes.loc[episodes["onset_day_ny"].eq(PILOT_DAY.date())]
else:
    scope = episodes.loc[episodes["within_study_window"]]


def placebo_of(ep):
    for weeks in PLACEBO_WEEKS:
        fake = pd.Series(ep._asdict())
        for column in ["incident_start", "onset_time", "alert_end"]:
            # Shifting the New York wall-clock time, which stays valid across daylight-saving changes
            fake[column] = localize_wall_clock(fake[column].tz_convert(LOCAL_TIMEZONE).tz_localize(None)
                                               + pd.Timedelta(weeks=weeks))
        quiet_routes = tuple(fake["disrupted_routes"]) or tuple(fake["routes"])
        if not route_disrupted(quiet_routes, fake["incident_start"] - PRE_WINDOW, fake["alert_end"] + POST_WINDOW):
            fake["episode_key"] = f"{ep.episode_key}|placebo{weeks}"
            return fake
    return None


calibration_source = scope.loc[scope["routes"].map(bool)]
calibration_source = calibration_source.sample(n=min(CALIBRATION_SAMPLE, len(calibration_source)), random_state=42)
pairs, without_placebo = [], []
for ep in calibration_source.itertuples(index=False):
    fake = placebo_of(ep)
    if fake is None:
        without_placebo.append(ep)
    else:
        pairs.append((pd.Series(ep._asdict()), fake))
print("Episode / placebo pairs:", len(pairs), "| sampled episodes without a quiet placebo:", len(without_placebo))
if without_placebo:
    display(pd.DataFrame(without_placebo)["disrupted_routes"].explode().value_counts().rename("episodes").to_frame())

calibration_rows = []
for threshold in COMPLETE_LOSS_GRID:
    for pair_id, (real_ep, fake_ep) in enumerate(pairs):
        for kind, ep in (("real", real_ep), ("placebo", fake_ep)):
            info, _ = measure_episode(ep, threshold)
            calibration_rows.append({**info, "threshold": threshold, "kind": kind, "pair_id": pair_id,
                                     "period": broad_period(real_ep["period"])})
calibration = pd.DataFrame(calibration_rows)
measured_pairs = (calibration.loc[calibration["measure_status"].eq("measured")]
                  .groupby(["threshold", "pair_id"])["kind"].nunique().eq(2))
calibration = calibration.merge(measured_pairs[measured_pairs].reset_index()[["threshold", "pair_id"]],
                                on=["threshold", "pair_id"])

Episode / placebo pairs: 143 | sampled episodes without a quiet placebo: 7


,episodes
disrupted_routes,
F,5
FX,5
A,3
R,2
G,2
B,1
N,1
C,1
D,1


### Why I use different thresholds during the day and at night

Moving from the most sensitive rule to a stricter one removes many placebo flags without losing much daytime separation from real episodes. Nighttime behaves differently, where ordinary train frequency is lower.

That leads me to `period_3_5` as the primary rule and `single_3` as an alternative. I'll keep both definitions available rather than hide the threshold choice inside one final flag. The clusters of unusual placebo flags also help identify feed problems worth investigating.

Under `period_3_5`, a run of at least two empty bins needs at least five expected trains on weekdays during the day, or three at night or on weekends. The alternative `single_3` uses three in both periods. I chose these thresholds using train records and placebo windows before comparing pickups.


In [ ]:
def window_rates(frame):
    return pd.Series({
        "windows": len(frame),
        "access_loss": frame["access_lost_complexes_during_alert"].gt(0).mean(),
        "route_loss": frame["lost_complexes_during_alert"].gt(0).mean(),
        "partial_loss": frame["partial_lost_complexes_during_alert"].gt(0).mean(),
        "pre_alert_loss": (frame["lost_complexes_pre_alert"] + frame["partial_lost_complexes_pre_alert"]).gt(0).mean(),
        "diversion": frame["diverted_complexes_during_alert"].gt(0).mean(),
        # Per-series rates: flagged route x direction x complex series per measured series
        "series_route_loss_rate": frame["lost_series_during_alert"].sum() / frame["series"].sum(),
        "series_access_loss_rate": frame["access_lost_series_during_alert"].sum() / frame["series"].sum(),
    })


def bootstrap_excess(frame, column, draws=BOOTSTRAP_DRAWS, seed=0):
    """95% interval of (real share - placebo share) of windows with any flag, resampling pairs."""
    wide = frame.pivot_table(index="pair_id", columns="kind", values=column, aggfunc="first").gt(0)
    if len(wide) < 2:
        return pd.Series({"excess_low": np.nan, "excess_high": np.nan})
    rng = np.random.default_rng(seed)
    samples = rng.integers(0, len(wide), size=(draws, len(wide)))
    excess = wide["real"].to_numpy()[samples].mean(axis=1) - wide["placebo"].to_numpy()[samples].mean(axis=1)
    return pd.Series({"excess_low": np.quantile(excess, 0.025), "excess_high": np.quantile(excess, 0.975)})


summary_rows = []
for (threshold, period), frame in [*calibration.groupby(["threshold", "period"]),
                                   *[((t, "all"), f) for t, f in calibration.groupby("threshold")]]:
    rates = pd.DataFrame({kind: window_rates(group) for kind, group in frame.groupby("kind")}).T
    row = {"threshold": threshold, "period": period}
    for kind in ["real", "placebo"]:
        if kind in rates.index:
            row.update({f"{metric}_{kind}": value for metric, value in rates.loc[kind].items()})
    interval = bootstrap_excess(frame, "access_lost_complexes_during_alert")
    row.update({"access_loss_excess": row.get("access_loss_real", np.nan) - row.get("access_loss_placebo", np.nan),
                "access_excess_low": interval["excess_low"], "access_excess_high": interval["excess_high"]})
    summary_rows.append(row)

calibration_summary = pd.DataFrame(summary_rows).sort_values(["period", "threshold"])
display(calibration_summary[["period", "threshold", "windows_real",
                             "access_loss_real", "access_loss_placebo", "access_loss_excess", "access_excess_low", "access_excess_high",
                             "series_access_loss_rate_real", "series_access_loss_rate_placebo",
                             "route_loss_real", "route_loss_placebo", "partial_loss_real", "partial_loss_placebo",
                             "pre_alert_loss_real", "pre_alert_loss_placebo", "diversion_real", "diversion_placebo"]].round(3))

,period,threshold,windows_real,access_loss_real,access_loss_placebo,access_loss_excess,access_excess_low,access_excess_high,series_access_loss_rate_real,series_access_loss_rate_placebo,route_loss_real,route_loss_placebo,partial_loss_real,partial_loss_placebo,pre_alert_loss_real,pre_alert_loss_placebo,diversion_real,diversion_placebo
10,all,2.0,135.0,0.444,0.193,0.252,0.148,0.356,0.026,0.006,0.615,0.296,0.415,0.259,0.689,0.452,0.333,0.141
11,all,3.0,135.0,0.378,0.133,0.244,0.141,0.341,0.018,0.004,0.526,0.163,0.415,0.259,0.585,0.348,0.333,0.141
12,all,4.0,135.0,0.304,0.111,0.193,0.104,0.281,0.012,0.004,0.415,0.148,0.422,0.259,0.541,0.319,0.333,0.141
13,all,5.0,135.0,0.222,0.081,0.141,0.052,0.222,0.009,0.003,0.326,0.111,0.430,0.259,0.511,0.311,0.333,0.141
14,all,6.0,135.0,0.200,0.074,0.126,0.044,0.207,0.005,0.002,0.267,0.104,0.430,0.259,0.496,0.311,0.333,0.141
0,night_or_weekend,2.0,66.0,0.576,0.303,0.273,0.121,0.424,0.041,0.010,0.712,0.439,0.242,0.121,0.697,0.470,0.333,0.136
2,night_or_weekend,3.0,66.0,0.455,0.182,0.273,0.106,0.425,0.025,0.006,0.576,0.197,0.242,0.121,0.545,0.273,0.333,0.136
4,night_or_weekend,4.0,66.0,0.364,0.152,0.212,0.061,0.364,0.016,0.006,0.455,0.182,0.242,0.121,0.500,0.227,0.333,0.136
6,night_or_weekend,5.0,66.0,0.227,0.121,0.106,-0.030,0.242,0.010,0.004,0.364,0.136,0.242,0.121,0.439,0.212,0.333,0.136
8,night_or_weekend,6.0,66.0,0.197,0.106,0.091,-0.030,0.212,0.005,0.003,0.258,0.136,0.242,0.121,0.409,0.212,0.333,0.136


## 7. Running the rule across the study

With the working rules selected, I'll measure all study episodes. Checkpoints every 500 episodes let an interrupted run resume. The settings and code fingerprints matter because an old checkpoint should not be reused after changing what the measurement means.


In [ ]:
# The fingerprint covers every setting and the measurement code itself, so changed rules never reuse old checkpoints
measurement_code = "".join(repr((f.__code__.co_code, f.__code__.co_consts))
                           for f in [run_ids, flag_empty_runs, flag_partial_runs, measure_episode])
SETTINGS_FINGERPRINT = hashlib.sha256(json.dumps({
    "version": PIPELINE_VERSION, "mode": RUN_MODE, "mapping": MAPPING_FINGERPRINT,
    "code": hashlib.sha256(measurement_code.encode()).hexdigest(), "connect": str(PRE_ALERT_CONNECT),
    "feed_groups": FEED_GROUPS, "gaps": [str(d) for d in source_gap_utc_days],
    "info_outages": sorted(info_outage_windows["alert_id"].astype(str).unique().tolist()),
    "complete": COMPLETE_LOSS_MIN_EXPECTED, "variant": LOSS_THRESHOLD_VARIANT, "run_bins": LOSS_MIN_RUN_BINS, "partial": [PARTIAL_LOSS_SHARE, PARTIAL_LOSS_MIN_EXPECTED],
    "baseline_buffer": str(BASELINE_EXCLUSION_BUFFER), "background": BACKGROUND_MIN_EXPECTED, "served": NORMALLY_SERVED_SHARE, "diversion_min_trains": DIVERSION_MIN_TRAINS, "access_any_direction": ACCESS_IGNORES_DIRECTION,
    "windows": [str(PRE_WINDOW), str(REFERENCE_PERIOD), str(POST_WINDOW), str(MAX_MEASURE_WINDOW), str(INCIDENT_START_MAX_LEAD)],
    "baseline": [BASELINE_WEEKS, BASELINE_SEARCH_WEEKS, MIN_BASELINE_DAYS], "feed": [FEED_OK_RATIO, FEED_MIN_EXPECTED],
}, sort_keys=True).encode()).hexdigest()[:12]
CHECKPOINT_DIR = SILVER_DIR / "_checkpoints" / SETTINGS_FINGERPRINT
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BIN_COLUMNS = ["episode_key", *KEY, "bin_utc", "phase", "trains", "expected", "all_route_trains", "all_route_expected",
               *FLAGS, "background_loss"]

# Resuming from earlier chunks measured with the same settings
done = {}
for path in sorted(CHECKPOINT_DIR.glob("chunk_*.pkl")):
    with open(path, "rb") as handle:
        chunk = pickle.load(handle)
    for key in chunk["measures"]["episode_key"]:
        done[key] = path
print(f"Settings {SETTINGS_FINGERPRINT}: {len(done)} episodes already measured")

# Date order lets neighboring episodes reuse the same cached days of train counts
remaining = scope.loc[~scope["episode_key"].isin(done)].sort_values("onset_time")
chunk_number = len(list(CHECKPOINT_DIR.glob("chunk_*.pkl")))
for chunk_start in range(0, len(remaining), CHECKPOINT_EVERY):
    measures, complexes, bins, service = [], [], [], []
    for ep in remaining.iloc[chunk_start: chunk_start + CHECKPOINT_EVERY].itertuples(index=False):
        try:
            info, grid = measure_episode(ep)
        except Exception as error:  # one bad episode must not stop the run
            info, grid = {"episode_key": ep.episode_key, "measure_status": "error",
                          "error": f"{type(error).__name__}: {str(error)[:300]}"}, None
        measures.append(info)
        if grid is not None:
            bins.append(grid.loc[grid[FLAGS].any(axis=1) | grid["background_loss"], BIN_COLUMNS])
            usable = grid.loc[grid["attributable"]]
            service.append(usable.assign(hour_utc=usable["bin_utc"].dt.floor("h"), during=usable["phase"].eq("during_alert"))
                           .groupby(["episode_key", "route_id", "station_complex_id", "hour_utc"])
                           .agg(trains=("trains", "sum"), expected=("expected", "sum"), during=("during", "any"))
                           .reset_index())
            complexes.append(
                grid.assign(**{f"{f}_bins": grid[f].astype(int) for f in FLAGS},
                            diverted_trains=grid["trains"].where(grid["diverted"], 0))
                .groupby(["episode_key", *KEY, "phase"])
                .agg(trains=("trains", "sum"), expected=("expected", "sum"), normally_served=("normally_served", "first"),
                     **{f"{f}_bins": (f"{f}_bins", "sum") for f in FLAGS}, diverted_trains=("diverted_trains", "sum"))
                .reset_index())
    chunk = {"measures": pd.DataFrame(measures),
             "complexes": pd.concat(complexes, ignore_index=True) if complexes else pd.DataFrame(),
             "bins": pd.concat(bins, ignore_index=True) if bins else pd.DataFrame(columns=BIN_COLUMNS),
             "service": pd.concat(service, ignore_index=True) if service else pd.DataFrame()}
    with open(CHECKPOINT_DIR / f"chunk_{chunk_number:05d}.pkl", "wb") as handle:
        pickle.dump(chunk, handle)
    chunk_number += 1
    print(f"Measured {min(chunk_start + CHECKPOINT_EVERY, len(remaining))} / {len(remaining)} remaining episodes")

chunks = []
for path in sorted(CHECKPOINT_DIR.glob("chunk_*.pkl")):
    with open(path, "rb") as handle:
        chunks.append(pickle.load(handle))
episode_measures = pd.concat([c["measures"] for c in chunks], ignore_index=True)
episode_measures = episode_measures.loc[episode_measures["episode_key"].isin(scope["episode_key"])].drop_duplicates("episode_key", keep="last")
complex_parts = [c["complexes"] for c in chunks if not c["complexes"].empty]
episode_complex_measured = pd.concat(complex_parts, ignore_index=True) if complex_parts else pd.DataFrame()
service_parts = [c["service"] for c in chunks if not c.get("service", pd.DataFrame()).empty]
service_hours = (pd.concat(service_parts, ignore_index=True) if service_parts
                 else pd.DataFrame(columns=["episode_key", "route_id", "station_complex_id", "hour_utc", "trains", "expected", "during"]))
service_hours = service_hours.loc[service_hours["episode_key"].isin(episode_measures["episode_key"])].copy()
service_hours["hour_utc"] = pd.to_datetime(service_hours["hour_utc"], utc=True)
bin_parts = [c["bins"] for c in chunks if not c["bins"].empty]
measured_bins = pd.concat(bin_parts, ignore_index=True) if bin_parts else pd.DataFrame(columns=BIN_COLUMNS)
for flag in [*FLAGS, "background_loss"]:
    measured_bins[flag] = measured_bins[flag].astype(bool)
measured_bins["bin_utc"] = pd.to_datetime(measured_bins["bin_utc"], utc=True)
for column in ["trains", "expected", "all_route_trains", "all_route_expected"]:
    measured_bins[column] = pd.to_numeric(measured_bins[column])
measured_bins = measured_bins.loc[measured_bins["episode_key"].isin(episode_measures["episode_key"])]

display(episode_measures["measure_status"].value_counts().to_frame("episodes"))
if episode_measures["measure_status"].eq("error").any():
    display(episode_measures.loc[episode_measures["measure_status"].eq("error"), ["episode_key", "error"]].head(20))

### Which episode gets credit when the windows overlap?

Two incident windows can inspect the same station at the same time. Without an attribution rule, one loss could enter twice. I'll assign each flagged bin to an active episode where possible, otherwise to the nearest onset, and retain that choice as part of the design.


In [ ]:
# One episode per flagged bin: where windows of several episodes overlap, the bin goes to the episode whose
# alert was active at that time, otherwise to the episode with the nearest onset
onset_by_episode = episodes.set_index("episode_key")["onset_time"]
attributable = measured_bins.loc[measured_bins[FLAGS].any(axis=1)].copy()
attributable["priority"] = attributable["phase"].ne("during_alert").astype(int)
attributable["distance"] = (attributable["bin_utc"] - attributable["episode_key"].map(onset_by_episode)).abs()
attributable["claims"] = attributable.groupby([*KEY, "bin_utc"])["episode_key"].transform("nunique")
attributed_bins = (attributable.sort_values(["priority", "distance"])
                   .drop_duplicates([*KEY, "bin_utc"], keep="first")
                   .drop(columns=["priority", "distance"]))
attributed_bins["shared_with_other_episode"] = attributed_bins["claims"].gt(1)
# The same attribution for hourly train counts: each route x complex x hour belongs to one episode
service_hours = service_hours.assign(
    priority=(~service_hours["during"].astype(bool)).astype(int),
    distance=(service_hours["hour_utc"] - service_hours["episode_key"].map(onset_by_episode)).abs())
attributed_service = (service_hours.sort_values(["priority", "distance"])
                      .drop_duplicates(["route_id", "station_complex_id", "hour_utc"], keep="first")
                      .drop(columns=["priority", "distance"]))
background_bins = (measured_bins.loc[measured_bins["background_loss"]]
                   .drop_duplicates([*KEY, "bin_utc"]))[[*KEY, "bin_utc"]]

print("Flagged bins:", len(attributable), "| after attribution:", len(attributed_bins),
      "| claimed by more than one episode:", int(attributed_bins["shared_with_other_episode"].sum()))

Flagged bins: 706683 | after attribution: 614766 | claimed by more than one episode: 81532


### What the trains say about each type of alert

I compare the measured patterns with the alert types. Suspensions should usually look different from reroutes, while skipped stops may leave other service available. Broad agreement is a useful check, and disagreements point to episodes worth reviewing rather than automatically prove one source wrong.


In [ ]:
# Episode counts after attribution (raw counts from measure_episode stay alongside)
attributed_counts = pd.DataFrame({
    f"attributed_{flag}_complexes_{phase}": attributed_bins.loc[attributed_bins[flag] & attributed_bins["phase"].eq(phase)]
    .groupby("episode_key")["station_complex_id"].nunique()
    for flag in FLAGS for phase in ["pre_alert", "during_alert", "post_alert"]
})
attributed_counts["attributed_first_flag_utc"] = attributed_bins.groupby("episode_key")["bin_utc"].min()
attributed_counts["attributed_last_flag_utc"] = attributed_bins.groupby("episode_key")["bin_utc"].max() + BIN
attributed_counts = attributed_counts.rename_axis("episode_key").reset_index()

disruption_episodes = (episodes.merge(episode_measures, on="episode_key", how="left", validate="one_to_one")
                       .merge(attributed_counts, on="episode_key", how="left", validate="one_to_one"))
count_columns = [c for c in attributed_counts.columns if c.endswith("_alert")]
measured_mask = disruption_episodes["measure_status"].eq("measured")
disruption_episodes.loc[measured_mask, count_columns] = disruption_episodes.loc[measured_mask, count_columns].fillna(0)

measured = disruption_episodes.loc[measured_mask].copy()
for flag in FLAGS:
    measured[f"has_{flag}"] = measured[f"attributed_{flag}_complexes_during_alert"].gt(0)
measured["loss_before_alert"] = (measured["attributed_lost_complexes_pre_alert"]
                                 + measured["attributed_partial_lost_complexes_pre_alert"]).gt(0)
measured["flag_starts_minus_onset_min"] = (measured["attributed_first_flag_utc"] - measured["onset_time"]).dt.total_seconds() / 60

display(measured.groupby(["primary_onset_effect", "cause_eligibility"])
        .agg(episodes=("episode_key", "size"), with_route_loss=("has_lost", "sum"),
             with_partial_loss=("has_partial_lost", "sum"), with_access_loss=("has_access_lost", "sum"),
             with_diversion=("has_diverted", "sum"), loss_before_alert=("loss_before_alert", "sum"),
             median_baseline_days=("baseline_days", "median"))
        .reset_index())
display(measured.groupby("period")["flag_starts_minus_onset_min"].describe())
print("Episodes whose pre-alert loss starts right at the end of the reference period:",
      int(measured["pre_alert_loss_starts_at_reference_end"].sum()), "of", int(measured["loss_before_alert"].sum()))
display(measured[[c for c in measured.columns if c.startswith("baseline_rejected_")]].sum().to_frame("baseline_days_rejected"))

,primary_onset_effect,cause_eligibility,episodes,with_route_loss,with_partial_loss,with_access_loss,with_diversion,loss_before_alert,median_baseline_days
0,full_suspension,excluded,17,14,9,10,5,11,6.0
1,full_suspension,main,55,42,38,27,20,32,6.0
2,full_suspension,robustness,26,20,16,18,11,15,6.0
3,partial_suspension,excluded,86,54,26,46,15,43,6.0
4,partial_suspension,main,843,492,544,379,243,569,6.0
5,partial_suspension,robustness,386,243,256,200,120,258,6.0
6,reroute,excluded,261,161,40,127,198,128,6.0
7,reroute,main,1199,617,728,413,856,797,6.0
8,reroute,robustness,363,203,219,152,271,242,6.0
9,severe_delay,excluded,43,27,38,22,19,35,6.0


,count,mean,std,min,25%,50%,75%,max
period,,,,,,,,
am_peak,898.0,-28.459911,36.279343,-138.0,-49.0,-28.0,-10.0,234.0
evening,671.0,-36.618480,37.126455,-148.0,-61.0,-37.0,-14.0,139.0
late_night,946.0,-32.914376,49.062276,-198.0,-63.0,-30.0,-6.0,272.0
midday,1297.0,-41.693138,37.105496,-207.0,-66.0,-41.0,-18.0,99.0
pm_peak,963.0,-31.388370,39.182747,-143.0,-56.0,-33.0,-11.0,190.0
weekend_day,833.0,-33.033613,41.223521,-180.0,-56.0,-32.0,-11.0,238.0
weekend_night,389.0,-30.035990,47.402355,-151.0,-57.0,-30.0,-9.0,181.0


Episodes whose pre-alert loss starts right at the end of the reference period: 291 of 4039


,baseline_days_rejected
baseline_rejected_holiday,2608.0
baseline_rejected_own_disruption,28030.0
baseline_rejected_feed,1711.0
baseline_rejected_missing_data,0.0


This helper puts the early flags in station and time order so I can review them directly. The June 30 example comes before the July study window; it illustrates the method rather than contributing to the study result.


In [ ]:
def inspect_pre_alert(episode_key):
    """Pre-alert flags of one episode, in station and time order."""
    rows = attributed_bins.loc[attributed_bins["episode_key"].eq(episode_key) & attributed_bins["phase"].eq("pre_alert")]
    if rows.empty:
        print(f"No pre-alert flags for {episode_key}.")
        return
    display(rows.assign(clock=rows["bin_utc"].dt.tz_convert(LOCAL_TIMEZONE).dt.strftime("%H:%M"),
                        station=rows["station_complex_id"].map(complex_names))
            [["route_id", "direction", "station", "clock", "trains", "expected", *FLAGS]]
            .sort_values(["route_id", "direction", "clock", "station"]))


inspect_pre_alert(f"{PILOT_EVENT_ID}-1")

## 8. How do I turn station-level changes into zone-hour measures?

The train measurements are at stations and short intervals, but the pickup outcome is at taxi zones and hours. I'll now aggregate exposure to that level while keeping its different forms and quality flags.


### Measured exposure

I'll retain minutes of loss, train deficits, and diversions rather than compress everything into a yes/no flag. Ridership weights help distinguish a loss at a normally busy time from one at a quiet time. They still describe potential intensity, not a count of displaced riders or people unable to board. Background loss stays separate so it does not become normal service by default.


In [ ]:
episode_attributes = disruption_episodes[["episode_key", "cause", "cause_eligibility", "primary_effect",
                                          "primary_onset_effect", "period", "is_right_censored"]]
bridge = station_complex_zone_bridge[["station_complex_id", "LocationID"]]
zone_bins = (attributed_bins.merge(bridge, on="station_complex_id", how="inner", validate="many_to_many")
             .merge(episode_attributes, on="episode_key", how="left", validate="many_to_one"))
zone_bins["hour_utc"] = zone_bins["bin_utc"].dt.floor("h")
MEASURES = {"lost": "route_lost", "partial_lost": "partial_lost", "access_lost": "access_lost"}
BIN_MINUTES = BIN.total_seconds() / 60


def hourly_minutes(rows, keys):
    """Minutes per hour with at least one flagged complex: each zone x bin counts once."""
    return (rows.drop_duplicates([*keys, "bin_utc"]).groupby([*keys, "hour_utc"]).size().mul(BIN_MINUTES))


# Measured exposure per zone, hour, and cause group (plus "all", deduplicated across groups)
parts = []
for group_label, rows in [*zone_bins.groupby("cause_eligibility"), ("all", zone_bins)]:
    columns = {}
    for flag, name in MEASURES.items():
        flagged = rows.loc[rows[flag]]
        columns[f"{name}_minutes"] = hourly_minutes(flagged, ["LocationID"])
        columns[f"{name}_minutes_during_alert"] = hourly_minutes(flagged.loc[flagged["phase"].eq("during_alert")], ["LocationID"])
    diverted = rows.loc[rows["diverted"]]
    columns["diverted_trains"] = diverted.groupby(["LocationID", "hour_utc"])["trains"].sum()
    columns["episodes"] = rows.groupby(["LocationID", "hour_utc"])["episode_key"].nunique()
    parts.append(pd.DataFrame(columns).fillna(0).reset_index().assign(cause_eligibility=group_label))
hourly_zone_measured = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["LocationID", "hour_utc", "cause_eligibility"])
for column in [*[f"{name}_minutes{suffix}" for name in MEASURES.values() for suffix in ("", "_during_alert")],
               "diverted_trains", "episodes"]:
    if column not in hourly_zone_measured:
        hourly_zone_measured[column] = 0.0

# Train deficit and ridership-weighted exposure proxies, per zone, hour, and cause group.
# Within each group, trains are summed per station complex and hour before weighting, so a complex's
# ridership is used once per hour (not once per overlapping episode). These are exposure proxies, not
# counts of people who lost service: the deficit covers the measured routes, while ridership covers the whole complex.
service_with_group = attributed_service.merge(episode_attributes[["episode_key", "cause_eligibility"]],
                                              on="episode_key", how="left")
service_parts = []
for group_label, rows in [*service_with_group.groupby("cause_eligibility"), ("all", service_with_group)]:
    if rows.empty:
        continue
    complex_hours = rows.groupby(["station_complex_id", "hour_utc"])[["trains", "expected"]].sum().reset_index()
    complex_hours["deficit"] = (complex_hours["expected"] - complex_hours["trains"]).clip(lower=0)
    complex_hours["deficit_share"] = (complex_hours["deficit"] / complex_hours["expected"].where(complex_hours["expected"] > 0)
                                      ).clip(upper=1).fillna(0)
    complex_hours["normal_riders"] = riders_for(complex_hours["station_complex_id"], complex_hours["hour_utc"])
    complex_hours["ridership_weighted_deficit"] = complex_hours["deficit_share"] * complex_hours["normal_riders"]
    service_parts.append(complex_hours.merge(bridge, on="station_complex_id", how="inner", validate="many_to_many")
                         .groupby(["LocationID", "hour_utc"])
                         .agg(expected_trains=("expected", "sum"), observed_trains=("trains", "sum"),
                              train_deficit=("deficit", "sum"), ridership_weighted_deficit=("ridership_weighted_deficit", "sum"))
                         .reset_index().assign(cause_eligibility=group_label))
hourly_zone_service = (pd.concat(service_parts, ignore_index=True) if service_parts else
                       pd.DataFrame(columns=["LocationID", "hour_utc", "expected_trains", "observed_trains",
                                             "train_deficit", "ridership_weighted_deficit", "cause_eligibility"]))

# Riders who normally enter a complex during bins when it had no service in at least one direction
# (an exposure proxy: directional access loss does not mean the whole complex had no service)
access_rows = zone_bins.loc[zone_bins["access_lost"]]
if len(access_rows):
    access_parts = []
    for group_label, rows in [*access_rows.groupby("cause_eligibility"), ("all", access_rows)]:
        rows = rows.drop_duplicates(["station_complex_id", "LocationID", "bin_utc"])
        rows = rows.assign(bin_riders=riders_for(rows["station_complex_id"], rows["bin_utc"]) / 4)
        access_parts.append(rows.groupby(["LocationID", "hour_utc"])["bin_riders"].sum()
                            .rename("ridership_weighted_access_loss").reset_index().assign(cause_eligibility=group_label))
    hourly_zone_measured = hourly_zone_measured.merge(pd.concat(access_parts, ignore_index=True),
                                                      on=["LocationID", "cause_eligibility", "hour_utc"], how="left")
else:
    hourly_zone_measured["ridership_weighted_access_loss"] = 0.0
hourly_zone_service["hour_utc"] = pd.to_datetime(hourly_zone_service["hour_utc"], utc=True)
hourly_zone_measured["hour_utc"] = pd.to_datetime(hourly_zone_measured["hour_utc"], utc=True)
hourly_zone_measured["LocationID"] = hourly_zone_measured["LocationID"].astype("int64")
hourly_zone_service["LocationID"] = hourly_zone_service["LocationID"].astype("int64")
hourly_zone_measured = hourly_zone_measured.merge(hourly_zone_service, on=["LocationID", "cause_eligibility", "hour_utc"], how="outer")
value_columns = [c for c in hourly_zone_measured.columns if c not in ("LocationID", "cause_eligibility", "hour_utc")]
hourly_zone_measured[value_columns] = hourly_zone_measured[value_columns].fillna(0)

# Background losses (planned work and other pre-existing changes) as their own exposure
background_zone = (background_bins.merge(bridge, on="station_complex_id", how="inner")
                   .assign(hour_utc=lambda f: f["bin_utc"].dt.floor("h")))
hourly_zone_background = (hourly_minutes(background_zone, ["LocationID"]).rename("background_route_lost_minutes")
                          .reset_index() if len(background_zone) else
                          pd.DataFrame(columns=["LocationID", "hour_utc", "background_route_lost_minutes"]))

if len(hourly_zone_measured):
    minute_columns = [c for c in hourly_zone_measured.columns if c.endswith("minutes") or c.endswith("during_alert")]
    assert hourly_zone_measured[minute_columns].le(60).all().all()
    display(hourly_zone_measured.groupby("cause_eligibility")[minute_columns + ["diverted_trains", "train_deficit",
                                                                                 "ridership_weighted_deficit"]].sum())

,route_lost_minutes,route_lost_minutes_during_alert,partial_lost_minutes,partial_lost_minutes_during_alert,access_lost_minutes,access_lost_minutes_during_alert,diverted_trains,train_deficit,ridership_weighted_deficit
cause_eligibility,,,,,,,,,
all,2531910.0,1466265.0,2528235.0,1471065.0,942675.0,508290.0,141130.0,2187624.5,9.098523e+07
excluded,409335.0,239445.0,147735.0,100485.0,154890.0,80910.0,11619.0,211693.5,7.117351e+06
main,1522140.0,856350.0,1737540.0,984030.0,544680.0,284925.0,95184.0,1449243.5,6.799155e+07
robustness,635730.0,380700.0,694755.0,407265.0,252750.0,144375.0,34327.0,566173.0,2.975463e+07


### Announced exposure

I'll also build the simpler announcement-based measure. It answers what riders were told about routes serving the zone, whereas measured exposure asks what the train records suggest changed. Comparing them later will show how much the result depends on that distinction.


In [ ]:
# Announced exposure: active alert states on a disrupted route, spread over the zones that route serves in the daytime
STATION_LIST_SYMBOL = {"6X": "6", "7X": "7", "FX": "F", "GS": "S", "FS": "S", "H": "S", "SI": "SIR", "SS": "SIR"}
route_zone_ids = {
    route: set(station_complex_zone_bridge.loc[
        station_complex_zone_bridge["station_complex_id"].isin(route_complexes.get(STATION_LIST_SYMBOL.get(route, route), set())),
        "LocationID"])
    for route in realtime_route_ids
}

active_states = (episodes.loc[episodes["episode_key"].isin(scope["episode_key"]),
                              ["episode_key", "active_alert_ids", "disrupted_routes", "cause_eligibility"]]
                 .explode("active_alert_ids").rename(columns={"active_alert_ids": "alert_id"})
                 .astype({"alert_id": "int64"})
                 .merge(candidate_history[["alert_id", "date", "next_update_time"]], on="alert_id", how="left", validate="many_to_one"))
active_states["end_observed"] = active_states["next_update_time"].notna()
active_states["start"] = active_states["date"]
active_states["end"] = active_states["next_update_time"].fillna(active_states["date"] + MAX_STATE_AGE)

announced = (active_states.explode("disrupted_routes").dropna(subset=["disrupted_routes"])
             .assign(LocationID=lambda f: f["disrupted_routes"].map(lambda r: sorted(route_zone_ids.get(r, set()))))
             .explode("LocationID").dropna(subset=["LocationID"]).astype({"LocationID": "int64"}))

announced_known = announced.loc[announced["end_observed"]]
hourly_zone_announced = pd.concat([
    intervals_to_hourly(announced_known, ["LocationID", "cause_eligibility"]),
    intervals_to_hourly(announced_known.assign(cause_eligibility="all"), ["LocationID", "cause_eligibility"]),
], ignore_index=True).rename(columns={"minutes": "announced_minutes"})
uncertain_announced_windows = announced.loc[~announced["end_observed"], ["episode_key", "LocationID", "start", "end"]]

print("Announced zone-hours:", len(hourly_zone_announced), "| Censored announced windows:", len(uncertain_announced_windows))

Announced zone-hours: 871851 | Censored announced windows: 20439


## 9. Which tables will I need to study pickups?

The pieces now come together in two tables. One gives the full hourly zone panel; the other keeps episode timing and zone details. Both are needed to explain how a particular disruption enters the later comparison.


### The hourly panel

The hourly panel needs a clear meaning for zero. Where the sources are missing or unreliable, I'll carry an unknown-exposure flag and its reason instead. I also flag daylight-saving hours, since a naive local hour is not always an ordinary one-hour interval. Those fields need to survive the pickup join.


In [ ]:
# Hourly panel: every subway-access zone x every New York local hour (naive, like TLC pickup times)
if RUN_MODE == "pilot":
    panel_start, panel_end = PILOT_DAY.tz_localize(None), PILOT_DAY.tz_localize(None) + pd.Timedelta(days=2)
else:
    panel_start, panel_end = STUDY_START.tz_localize(None), STUDY_END.tz_localize(None)
panel_hours = pd.date_range(panel_start, panel_end, freq="h", inclusive="left")
panel = pd.MultiIndex.from_product([subway_zones["LocationID"], panel_hours], names=["LocationID", "hour_local"]).to_frame(index=False)
panel = panel.merge(subway_zones, on="LocationID", how="left", validate="many_to_one")


def to_panel_hours(frame, value_columns, keys=("LocationID",)):
    """UTC-hour rows to naive local hours; the repeated fall-back hour is combined and capped at 60 minutes."""
    if frame.empty:
        return pd.DataFrame(columns=[*keys, "hour_local", *value_columns])
    out = frame.assign(hour_local=to_local_naive(frame["hour_utc"]))
    out = out.groupby([*keys, "hour_local"])[list(value_columns)].sum().reset_index()
    minute_columns = [c for c in value_columns if "minutes" in c]
    out[minute_columns] = out[minute_columns].clip(upper=60)
    return out


# Measured and announced exposure, one column per cause group
for source, values in [(hourly_zone_measured, [c for c in hourly_zone_measured.columns
                                               if c not in ("LocationID", "hour_utc", "cause_eligibility")]),
                       (hourly_zone_announced, ["announced_minutes"])]:
    if source.empty:
        continue
    hourly = to_panel_hours(source, values, keys=("LocationID", "cause_eligibility"))
    wide = hourly.pivot_table(index=["LocationID", "hour_local"], columns="cause_eligibility", values=values, aggfunc="sum")
    wide.columns = [f"{value}_{group}" for value, group in wide.columns]
    panel = panel.merge(wide.reset_index(), on=["LocationID", "hour_local"], how="left")
panel = panel.merge(to_panel_hours(hourly_zone_background, ["background_route_lost_minutes"]),
                    on=["LocationID", "hour_local"], how="left")
exposure_columns = [c for c in panel.columns if c not in ("LocationID", "hour_local", "zone_borough", "zone_name")]
panel[exposure_columns] = panel[exposure_columns].fillna(0)

# Normal subway riders per zone-hour (all complexes in the zone): a weight and control for every hour
zone_normal = (normal_riders.merge(bridge.assign(station_complex_id=bridge["station_complex_id"].astype(str)),
                                   on="station_complex_id", how="inner")
               .groupby(["LocationID", "quarter", "day_type", "hour"])["normal_riders"].sum()
               .rename("normal_subway_riders").reset_index())
panel["quarter"] = panel["hour_local"].dt.to_period("Q").astype(str)
panel["day_type"] = day_type(panel["hour_local"]).to_numpy()
panel["hour"] = panel["hour_local"].dt.hour
panel = panel.merge(zone_normal, on=["LocationID", "quarter", "day_type", "hour"], how="left")
panel["normal_subway_riders"] = panel["normal_subway_riders"].fillna(0)
panel = panel.drop(columns=["quarter", "hour"])

# Staten Island Railway fares are collected only at the ferry terminal, so most SIR stations have no ridership:
# flag zones where any station complex lacks ridership, so zero riders there is not read as "no riders"
complexes_with_ridership = set(normal_riders["station_complex_id"])
zones_missing_ridership = set(bridge.loc[~bridge["station_complex_id"].astype(str).isin(complexes_with_ridership), "LocationID"])
panel["ridership_complete"] = ~panel["LocationID"].isin(zones_missing_ridership)
print("Zones with a station complex lacking ridership:", len(zones_missing_ridership))

# Daylight saving: the fall-back hour label covers two real hours; the spring-forward hour label covers none
panel_hours_index = pd.DatetimeIndex(panel_hours)
ambiguous_hours = set(panel_hours_index[panel_hours_index.tz_localize(LOCAL_TIMEZONE, ambiguous="NaT", nonexistent="shift_forward").isna()])
nonexistent_hours = set(panel_hours_index[panel_hours_index.tz_localize(LOCAL_TIMEZONE, ambiguous=True, nonexistent="NaT").isna()])
panel["ambiguous_dst_hour"] = panel["hour_local"].isin(ambiguous_hours)
panel["nonexistent_dst_hour"] = panel["hour_local"].isin(nonexistent_hours)

# Reasons the exposure is not known
day = panel["hour_local"].dt.date
gap_hours = intervals_to_hourly(alert_gap_intervals.assign(key=1), ["key"]) if len(alert_gap_intervals) else pd.DataFrame(columns=["hour_utc"])
gap_local_hours = set(to_local_naive(gap_hours["hour_utc"])) if len(gap_hours) else set()
panel["alert_gap"] = panel["hour_local"].isin(gap_local_hours) | day.isin(alert_partial_gap_days)
panel["train_data_missing"] = ~panel["hour_local"].dt.strftime("%Y-%m-%d").isin(train_available_days)

route_groups_by_zone = (pd.DataFrame([(route, zone) for route, zones in route_zone_ids.items() for zone in zones],
                                     columns=["route_id", "LocationID"])
                        .assign(feed_group=lambda f: f["route_id"].map(ROUTE_FEED_GROUP).fillna("other"))
                        [["LocationID", "feed_group"]].drop_duplicates())
feed_problems = (feed_health.loc[~feed_health["feed_ok"]]
                 .assign(hour_local=lambda f: pd.to_datetime(f["day_ny"] + " " + f["clock"].str[:2] + ":00"))
                 [["feed_group", "hour_local"]].drop_duplicates()
                 .merge(route_groups_by_zone, on="feed_group")[["LocationID", "hour_local"]].drop_duplicates()
                 .assign(feed_problem=True))
panel = panel.merge(feed_problems, on=["LocationID", "hour_local"], how="left")
panel["feed_problem"] = panel["feed_problem"].eq(True)

unmeasured = disruption_episodes.loc[disruption_episodes["episode_key"].isin(scope["episode_key"])
                                     & disruption_episodes["measure_status"].ne("measured")]
unmeasured_windows = (unmeasured[["episode_key", "disrupted_routes", "incident_start", "alert_end"]]
                      .explode("disrupted_routes").dropna(subset=["disrupted_routes"])
                      .assign(LocationID=lambda f: f["disrupted_routes"].map(lambda r: sorted(route_zone_ids.get(r, set()))))
                      .explode("LocationID").dropna(subset=["LocationID"]).astype({"LocationID": "int64"})
                      .rename(columns={"incident_start": "start", "alert_end": "end"}))
for name, windows in [("unmeasured_episode", unmeasured_windows), ("censored_alert_state", uncertain_announced_windows)]:
    hours = intervals_to_hourly(windows, ["LocationID"]) if len(windows) else pd.DataFrame(columns=["LocationID", "hour_utc"])
    flagged = (hours.assign(hour_local=to_local_naive(hours["hour_utc"]))[["LocationID", "hour_local"]]
               .drop_duplicates().assign(**{name: True}) if len(hours) else pd.DataFrame(columns=["LocationID", "hour_local", name]))
    panel = panel.merge(flagged, on=["LocationID", "hour_local"], how="left")
    panel[name] = panel[name].eq(True)

REASONS = ["alert_gap", "train_data_missing", "feed_problem", "unmeasured_episode", "censored_alert_state"]
panel["exposure_known"] = ~panel[REASONS].any(axis=1)
hourly_panel = panel

display(pd.Series({"zone_hours": len(panel), "exposure_known_share": panel["exposure_known"].mean(),
                   **{f"{r}_share": panel[r].mean() for r in REASONS},
                   "ambiguous_dst_zone_hours": int(panel["ambiguous_dst_hour"].sum()),
                   "nonexistent_dst_zone_hours": int(panel["nonexistent_dst_hour"].sum())}, name="value").to_frame())

Zones with a station complex lacking ridership: 8


,value
zone_hours,2.890800e+06
exposure_known_share,8.950391e-01
alert_gap_share,5.331050e-02
train_data_missing_share,0.000000e+00
feed_problem_share,2.557562e-02
unmeasured_episode_share,1.493324e-02
censored_alert_state_share,1.738792e-02
ambiguous_dst_zone_hours,3.300000e+02
nonexistent_dst_zone_hours,3.300000e+02


### The event-study table

The episode-zone table keeps announced and measured starts separately. That lets the causal analysis choose and explain its timing anchor rather than assume the first alert and the first service loss happened together.


In [ ]:
# Event-study table: one row per episode x exposed zone, with measured and announced timing
zone_flags = zone_bins.copy()
first_last = {}
for flag, name in {**MEASURES, "diverted": "diverted"}.items():
    rows = zone_flags.loc[zone_flags[flag]].groupby(["episode_key", "LocationID"])["bin_utc"]
    first_last[f"first_{name}_utc"] = rows.min()
    first_last[f"{name}_complexes"] = zone_flags.loc[zone_flags[flag]].groupby(["episode_key", "LocationID"])["station_complex_id"].nunique()
    first_last[f"{name}_complex_minutes"] = (zone_flags.loc[zone_flags[flag]]
                                             .drop_duplicates(["episode_key", "LocationID", "station_complex_id", "bin_utc"])
                                             .groupby(["episode_key", "LocationID"]).size().mul(BIN_MINUTES))

if zone_flags.empty:
    event_study = pd.DataFrame()
else:
    event_study = pd.DataFrame(first_last)
    event_study["measured_start_utc"] = event_study[[c for c in event_study.columns if c.startswith("first_")]].min(axis=1)
    event_study["measured_end_utc"] = zone_flags.groupby(["episode_key", "LocationID"])["bin_utc"].max() + BIN
    event_study["shared_bins"] = zone_flags.groupby(["episode_key", "LocationID"])["shared_with_other_episode"].sum()
    event_study = event_study.reset_index()
    count_columns = [c for c in event_study.columns if c.endswith("_complexes") or c.endswith("_complex_minutes")]
    event_study[count_columns] = event_study[count_columns].fillna(0)
    event_study["treatment_tier"] = np.select(
        [event_study["access_lost_complexes"].gt(0), event_study["route_lost_complexes"].gt(0),
         event_study["partial_lost_complexes"].gt(0), event_study["diverted_complexes"].gt(0)],
        ["access_lost", "route_lost", "partial_lost", "diverted_only"], "none")
    event_study = (event_study
                   .merge(disruption_episodes[["episode_key", "event_id", "incident_start", "incident_start_uncapped", "onset_time",
                                               "alert_end", "is_right_censored", "censor_reason", "primary_effect",
                                               "primary_onset_effect", "period", "weekend_or_holiday", "cause",
                                               "cause_eligibility", "multiple_causes", "on_alert_gap_day", "baseline_days"]],
                          on="episode_key", how="left", validate="many_to_one")
                   .merge(zone_info, on="LocationID", how="left", validate="many_to_one"))
    event_study["measured_start_minus_onset_min"] = (event_study["measured_start_utc"] - event_study["onset_time"]).dt.total_seconds() / 60
    display(event_study.groupby(["treatment_tier", "cause_eligibility"]).size().unstack(fill_value=0))

cause_eligibility,excluded,main,robustness
treatment_tier,,,
access_lost,1900,7561,3209
diverted_only,2241,12960,4744
partial_lost,1323,22042,8067
route_lost,2390,11929,4753


In [ ]:
# Selecting weekday daytime episodes with main causes and service-loss exposure
daytime_periods = ["am_peak", "midday", "pm_peak"]
loss_tiers = ["access_lost", "route_lost", "partial_lost"]

eligible_episode_zones = event_study.loc[
    event_study["period"].isin(daytime_periods)
    & event_study["cause_eligibility"].eq("main")
    & event_study["treatment_tier"].isin(loss_tiers)
    & event_study["onset_time"].ge(STUDY_START_UTC)
    & event_study["onset_time"].lt(STUDY_END_UTC)
].copy()

# Checking that each episode–zone combination appears only once
assert not eligible_episode_zones.duplicated(
    ["episode_key", "LocationID"]
).any(), "Repeated episode–zone combinations found."

# Counting incidents separately from the number of exposed zones
disruption_sample_summary = (
    eligible_episode_zones
    .groupby(["zone_borough", "period"])
    .agg(
        events=("event_id", "nunique"),
        episodes=("episode_key", "nunique"),
        episode_zone_pairs=("episode_key", "size"),
        affected_zones=("LocationID", "nunique"),
        censored_episodes=(
            "episode_key",
            lambda keys: eligible_episode_zones.loc[
                keys.index
            ].loc[
                lambda frame: frame["is_right_censored"],
                "episode_key",
            ].nunique(),
        ),
    )
    .reindex(
        pd.MultiIndex.from_product(
            [sorted(subway_zones["zone_borough"].unique()), daytime_periods],
            names=["zone_borough", "period"],
        ),
        fill_value=0,
    )
    .reset_index()
)

display(disruption_sample_summary)

,zone_borough,period,events,episodes,episode_zone_pairs,affected_zones,censored_episodes
0,Bronx,am_peak,189,195,825,28,15
1,Bronx,midday,257,262,1121,28,16
2,Bronx,pm_peak,212,214,843,28,17
3,Brooklyn,am_peak,432,442,2460,48,28
4,Brooklyn,midday,595,602,3085,48,24
5,Brooklyn,pm_peak,418,426,2459,48,22
6,Manhattan,am_peak,501,517,3961,53,38
7,Manhattan,midday,648,659,4788,53,26
8,Manhattan,pm_peak,492,499,3853,53,29
9,Queens,am_peak,235,242,1044,27,21


## 10. What do I save from each run?

I save the hourly panel, episode tables, and run settings together. The settings help me trace a change in the results back to the sources or the measurement rules.

The prepared snapshot includes the saved measurement run used downstream. To rebuild it here, I need the original alert, train, and ridership inputs. The earlier export session showed a kernel crash, so I keep the saved-run file checks separate from whether that session finished cleanly.


In [ ]:
def parquet_safe(frame):
    output = frame.copy()
    for column in output.columns:
        if output[column].dtype == object and output[column].map(lambda v: isinstance(v, (tuple, set, frozenset))).any():
            output[column] = output[column].map(lambda v: list(v) if isinstance(v, (tuple, set, frozenset)) else v)
    for column in ["window_truncated", "pre_alert_loss_starts_at_reference_end"]:
        if column in output:
            output[column] = output[column].astype("boolean")
    return output


TABLE_DESCRIPTIONS = {
    "hourly_panel": "Subway-access zone x New York local hour (naive). Exposure minutes (0-60) by measure and cause group, "
                    "background minutes, and exposure_known with its reasons.",
    "event_study": "Episode x zone with any attributed flag: measured first times by type, measured end, alert onset/end, tier.",
    "disruption_episodes": "One row per alert episode: timing (UTC), effects, cause, routes, measurement summary.",
    "hourly_zone_measured": "Zone x UTC hour x cause group: minutes with route loss, partial loss, access loss; diverted trains; "
                            "expected and observed trains, train deficit, and ridership-weighted exposure proxies "
                            "(deficit share x normal complex riders; normal riders during directional access loss).",
    "hourly_zone_background": "Zone x UTC hour: minutes with pre-existing route loss (planned work etc.), not attributed.",
    "hourly_zone_announced": "Zone x UTC hour x cause group: minutes under an active alert on a route serving the zone.",
    "attributed_bins": "Flagged route x direction x complex x 15-minute bins, each assigned to one episode.",
    "episode_complex_measured": "Episode x route x direction x complex x phase: observed and expected trains, flag counts.",
    "uncertain_announced_windows": "Censored alert states x zone: announced exposure unknown.",
    "alert_coverage_daily": "Study day: alert updates, empty days, partial-day gaps.",
    "feed_health": "Day x clock bin x feed group: realtime feed status.",
    "unmapped_realtime_stops": "Realtime stop ids that are not passenger stations (dropped).",
    "station_complex_zone_bridge": "Station complex x taxi zone (a complex can span zones: do not sum exposure across zones).",
}

if SAVE_OUTPUTS:
    run_dir = SILVER_DIR / f"{RUN_MODE}_{LOSS_THRESHOLD_VARIANT}_{datetime.now():%Y%m%d_%H%M%S}"
    run_dir.mkdir(parents=True)
    tables = {
        "hourly_panel": hourly_panel, "event_study": event_study, "disruption_episodes": disruption_episodes,
        "hourly_zone_measured": hourly_zone_measured, "hourly_zone_background": hourly_zone_background,
        "hourly_zone_announced": hourly_zone_announced, "attributed_bins": attributed_bins,
        "episode_complex_measured": episode_complex_measured, "uncertain_announced_windows": uncertain_announced_windows,
        "alert_coverage_daily": alert_coverage_daily, "feed_health": feed_health,
        "unmapped_realtime_stops": unmapped_stops, "station_complex_zone_bridge": station_complex_zone_bridge,
    }
    for name, frame in tables.items():
        parquet_safe(frame).to_parquet(run_dir / f"{name}.parquet", index=False)  # empty tables are written too

    manifest = {
        "pipeline_version": PIPELINE_VERSION, "run_mode": RUN_MODE, "settings_fingerprint": SETTINGS_FINGERPRINT,
        "mapping_fingerprint": MAPPING_FINGERPRINT, "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "study_start": str(STUDY_START), "study_end_exclusive": str(STUDY_END), "alert_clock": "UTC",
        "settings": {
            "baseline_weeks": BASELINE_WEEKS, "baseline_search_weeks": BASELINE_SEARCH_WEEKS, "min_baseline_days": MIN_BASELINE_DAYS,
            "pre_window": str(PRE_WINDOW), "reference_period": str(REFERENCE_PERIOD), "post_window": str(POST_WINDOW),
            "incident_start_max_lead": str(INCIDENT_START_MAX_LEAD), "baseline_exclusion_buffer": str(BASELINE_EXCLUSION_BUFFER), "max_state_age": str(MAX_STATE_AGE),
            "loss_threshold_variant": LOSS_THRESHOLD_VARIANT,
            "complete_loss_min_expected": COMPLETE_LOSS_MIN_EXPECTED, "loss_min_run_bins": LOSS_MIN_RUN_BINS,
            "partial_loss_share": PARTIAL_LOSS_SHARE, "partial_loss_min_expected": PARTIAL_LOSS_MIN_EXPECTED,
            "background_min_expected": BACKGROUND_MIN_EXPECTED, "normally_served_share": NORMALLY_SERVED_SHARE,
            "diversion_min_trains": DIVERSION_MIN_TRAINS,
            "access_ignores_direction": ACCESS_IGNORES_DIRECTION, "feed_ok_ratio": FEED_OK_RATIO,
            "feed_min_expected": FEED_MIN_EXPECTED, "cause_eligibility": CAUSE_ELIGIBILITY,
            "manual_stop_parent": MANUAL_STOP_PARENT,
        },
        "inputs": {
            "stations_sha256": STATIONS_SHA256, "taxi_zone_sha256": TAXI_ZONE_SHA256,
            "alert_months": alert_month_log.assign(fetched_at_utc=alert_month_log["fetched_at_utc"].astype(str)).to_dict("records"),
            "train_days": {"first": f"{train_days.min():%Y-%m-%d}", "last": f"{train_days.max():%Y-%m-%d}",
                           "status": reduction_log["status"].value_counts().to_dict()},
        },
        "environment": {"python": platform.python_version(), "pandas": pd.__version__, "geopandas": gpd.__version__},
        "tables": {name: {"rows": len(frame), "description": TABLE_DESCRIPTIONS.get(name, "")} for name, frame in tables.items()},
    }
    (run_dir / "run_manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
    (SILVER_DIR / f"latest_{RUN_MODE}_{LOSS_THRESHOLD_VARIANT}.txt").write_text(run_dir.name, encoding="utf-8")
    display(pd.DataFrame([{"table": n, "rows": len(f)} for n, f in tables.items()]))
    print("Saved to", run_dir)

## What I take into the pickup analysis

I started with an alert that told me where to look. The train records helped me separate a route losing service from a station losing all service in a direction. That difference matters when I ask how many more pickups I see nearby. A rider who still has another train faces a different choice from someone with no train at that station.

I check individual cases in the [subway validation notebook](04_mta_disruptions_validation.ipynb). Then I use the measured loss windows in the [causal analysis](08_subway_fhv_causal_analysis_storytelling.ipynb) to compare completed pickups in affected zones with pickups in similar zones.


In [ ]:
from pathlib import Path
import json
import pandas as pd

def verify_saved_mta_run(run_directory):
    """Checking saved tables against their manifest and intended study period."""

    run_directory = Path(run_directory)
    manifest_path = run_directory / "run_manifest.json"
    assert manifest_path.is_file(), f"Missing manifest: {manifest_path}"

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

    # Checking every saved table against its recorded row count
    checks = []

    for name, metadata in manifest["tables"].items():
        path = run_directory / f"{name}.parquet"
        assert path.is_file(), f"Missing table: {path}"

        import pyarrow.parquet as pq

        actual_rows = pq.read_metadata(path).num_rows
        checks.append({
            "table": name,
            "manifest_rows": metadata["rows"],
            "saved_rows": actual_rows,
            "rows_match": actual_rows == metadata["rows"],
        })

    checks = pd.DataFrame(checks)
    display(checks)
    assert checks["rows_match"].all(), "Saved row counts differ from the manifest."

    # Checking the panel's actual dates and unique FHV join keys
    panel_keys = pd.read_parquet(
        run_directory / "hourly_panel.parquet",
        columns=["LocationID", "hour_local"],
    )

    start = pd.Timestamp("2024-07-01")
    end = pd.Timestamp("2026-07-01")

    display(pd.Series({
        "manifest_start": manifest["study_start"],
        "manifest_end_exclusive": manifest["study_end_exclusive"],
        "first_panel_hour": panel_keys["hour_local"].min(),
        "last_panel_hour": panel_keys["hour_local"].max(),
        "zones": panel_keys["LocationID"].nunique(),
        "duplicate_join_keys": int(
            panel_keys.duplicated(["LocationID", "hour_local"]).sum()
        ),
    }, name="saved_run").to_frame())

    assert pd.Timestamp(manifest["study_start"]).tz_localize(None) == start
    assert pd.Timestamp(manifest["study_end_exclusive"]).tz_localize(None) == end
    assert panel_keys["hour_local"].notna().all()
    assert panel_keys["LocationID"].notna().all()
    assert panel_keys["hour_local"].min() == start
    assert panel_keys["hour_local"].max() == end - pd.Timedelta(hours=1)
    assert not panel_keys.duplicated(["LocationID", "hour_local"]).any()

    expected_hours = pd.date_range(start, end, freq="h", inclusive="left")
    assert panel_keys["hour_local"].isin(expected_hours).all()
    assert len(panel_keys) == (
        panel_keys["LocationID"].nunique() * len(expected_hours)
    )

    return checks


saved_run_checks = verify_saved_mta_run(
    PROJECT_ROOT / "data/silver/mta_disruptions/full_period_3_5_20261012_034129"
)